# eu_and_v3 — single-switch report: Open, the expected longest open run of the stem

For each of the first `N_DESIGNS` designs of the NucSyn CSV this notebook estimates, by sampling
structures from the folding ensemble,

* **Open (switch + trigger)** = expected length of the **longest run of consecutive** open `stem_up` nt
  for the switch with its real trigger;
* **Open (switch alone)** = the same for the switch without any trigger;
* next to each, **E[k]** = expected number of open nt anywhere on the stem (consecutive or not);

checks the sampler against the exact base-pair-probability matrix, defines
**ON** = Open (switch + trigger) and **OFF** = Open (switch alone) as percentages of the stem, ranks the
designs by **ON - OFF** (percentage points), and also lists the ratio ON / OFF. It writes `var/reports/eu_and_v3_report_<timestamp>.html` (interactive, with a Details button per design) and `.pdf` (the table, then about three pages of details per design; the Details links of the table jump to them, with bookmarks) (plus a CSV
of every design) with the correlation between this ranking and the CSV's own rank.

The definitions are written out in the report itself.

**This notebook builds two reports in one run:** part 1 is the single-switch report (`eu_and_v3_report_<time>.pdf`) and part 2 is the AND report (`eu_and_v3_and_report_<time>.pdf`), built from the top `AND_TOP_N` designs of part 1. All parameters of both parts are in the one parameter cell.

## Setup

In [ ]:
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_fixtures.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break
for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_switch_construct.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Parameters — every tunable in one place

In [ ]:
# ---- data source ------------------------------------------------------------
CSV_PATH = (
    "/Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/"
    "output_T3/targets/sele/basic_switch/results_ranked.csv"
)
COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
    "trigger_start": "trigger_start_pos",
    "trigger_end": "trigger_end_pos",
}
CARRY_METRICS = ("sim_score", "mcc", "switch_toehold_unpaired_prob_mean", "energy_trigger_mfe")
N_DESIGNS = 150            # first rows of the CSV that are scored

# ---- folding and sampling ------------------------------------------------------
TEMPERATURE_C = 37.0
N_SAMPLES = 2000          # structures drawn per state
SEED = 1                  # ViennaRNA's generator is process-global; fixed so a rerun repeats
MAX_WORKERS = 6           # process-pool width
CHECK_SEED_OFFSET = 1000  # a second, independent sampling run (seed + this) only measures the noise

# ---- verification --------------------------------------------------------------
VERIFY_SIGMA = 4.0        # allowed disagreement with the matrix, in standard errors of the sampled mean
VERIFY_FLOOR_NT = 0.02    # plus this many nt (rounding of the matrix, rare draws)

# ---- report --------------------------------------------------------------------
CONTROL_EXTRA_TRIGGERS = ()  # optional extra RNA sequences offered as mismatched controls besides the other designs' triggers
N_CONTROLS = 5            # mismatched triggers per design; OFF is the worst of the alone state and these
CONTROL_MAX_OVERLAP_NT = 8  # the mismatched control trigger shares at most this many nt of identity with the real one
OFF_FLOOR_PCT = 1.0       # OFF (% of the stem) is not allowed below this in the ratio ON / OFF
REPORT_TOP_N = 20         # designs listed in the report table (the CSV export has all)

# Re-render from a saved results file instead of sampling again (None = sample).
RESULTS_TO_LOAD = None

# ---- AND report (part 2) --------------------------------------------------------
RESULTS_TO_LOAD_AND = None   # re-render the AND part from a saved results file instead of sampling (None = sample)
# ---- data source ------------------------------------------------------------
AND_TOP_N = 20            # top singles (by ON - OFF) fused into the N x N grid
# Where the pool comes from: "latest" = the newest eu_and_v3_all_designs_*.csv written by the single
# report (eu_and_v3.ipynb; run it first), so the pool is exactly that report's top AND_TOP_N designs
# (its tied group first, then by ON - OFF). A path uses that file; None scores the first N_SINGLE
# rows again here (alone-only OFF, no controls, no tie grouping).

# ---- AND construct -------------------------------------------------------------
CUT_5P_BEFORE = "kozak"   # the 5' half keeps everything before its OWN kozak
KEEP_3P_AFTER = "prefix"  # the 3' half keeps everything after its OWN prefix
SPACER_VARIANTS = ("no_spacer", "spacer")   # both are run and reported separately
SPACER_MIN_LENGTH = 15
SPACER_MAX_LENGTH = 20
SPACER_SEED = 1
SPACER_CANDIDATE_POOL_SIZE = 150
FAKE_MAX_SEQ_OVERLAP_NT = 8   # trigger A vs trigger B: longest identical / complementary run allowed
WOBBLE_MAX_FRACTION = 0.50    # max share of a counted complementary run that may be G.U wobble

# ---- folding and sampling ------------------------------------------------------

# ---- verification and ratio ----------------------------------------------------

# ---- report --------------------------------------------------------------------

# Re-render from a saved results file instead of sampling again (None = sample).

# Data for an interactive page (filters + expandable 2D structures and heat maps): rows of ALL candidates
# and the detail panels of the best ARTIFACT_DETAIL_N, written to var/reports/eu_and_v3_and_artifact_<time>/.
EXPORT_ARTIFACT = True
ARTIFACT_DETAIL_N = 300


# Part 1 — single-switch report

## Load the designs and sample both states

In [ ]:
import collections, datetime, html, json, math, statistics, subprocess, time
from pathlib import Path

from _joint_state_parallel import pick_fake_triggers, use_fold_engine
from _sampled_stem_parallel import DesignJob, run_parallel_designs
from _switch_construct import load_ranked_designs

folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C)
use_fold_engine(folder)   # the pool forks: publish the one engine first

_report_root = Path.cwd()
for _base in (_report_root, *_report_root.parents):
    if (_base / "pyproject.toml").exists():
        _report_root = _base / "var" / "reports"
        break
_report_root.mkdir(parents=True, exist_ok=True)

designs, problems = load_ranked_designs(
    CSV_PATH, folder, columns=COLUMNS, carry_metrics=CARRY_METRICS, limit=N_DESIGNS,
)
for _line_no, _reason in problems[:10]:
    print(f"  skipped line {_line_no}: {_reason}")
print(f"loaded {len(designs)} designs ({len(problems)} rows skipped)")

if RESULTS_TO_LOAD is None:
    # control: for each design a mismatched trigger, i.e. the real trigger of another design that
    # shares at most CONTROL_MAX_OVERLAP_NT nt of identical sequence with the true trigger
    _control_pool = designs + [{"rank": f"extra{_k}", "trigger": _t} for _k, _t in enumerate(CONTROL_EXTRA_TRIGGERS)]
    for _i, _d in enumerate(designs):
        _fakes = pick_fake_triggers((_d["trigger"],), {_d["rank"]}, _control_pool, N_CONTROLS,
                                    start_offset=_i + 1, max_overlap_nt=CONTROL_MAX_OVERLAP_NT)
        _d["control_triggers"] = tuple(_f["trigger"] for _f in _fakes)
        _d["control_ranks"] = [_f["rank"] for _f in _fakes]
    jobs = [
        DesignJob(
            key=_d["rank"], switch=_d["switch"], trigger=_d["trigger"],
            stem_up=_d["domains"]["stem_up"], stem_down=_d["domains"]["stem_down"],
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, seed=SEED, controls=_d["control_triggers"],
        )
        for _d in designs
    ]
    _t0 = time.time()
    results = run_parallel_designs(jobs, max_workers=MAX_WORKERS)
    check_results = run_parallel_designs(
        [j._replace(seed=SEED + CHECK_SEED_OFFSET) for j in jobs], max_workers=MAX_WORKERS)
    print(f"sampled {len(jobs)} designs twice (second run = noise check) in {time.time() - _t0:.0f}s")
    _results_path = _report_root / f"eu_and_v3_results_{datetime.datetime.now():%Y%m%dT%H%M%S}.json"
    _results_path.write_text(json.dumps({"results": results, "check": check_results}))
    print(f"wrote {_results_path}")
else:
    _saved = json.loads(Path(RESULTS_TO_LOAD).read_text())
    results, check_results = _saved["results"], _saved["check"]
    print(f"loaded results for {len(results)} designs from {RESULTS_TO_LOAD}")

## Open, checks and ranking

Per state: `run_nt_sampled` is the mean over the draws of the longest open run r (this is Open),
`open_nt_sampled` the mean number of open nt k, `open_nt_matrix` the exact mean number of open nt.
The matrix cannot give Open, so a state is *verified* when (1) the sampled mean k matches the matrix
within `VERIFY_SIGMA` standard errors (+ `VERIFY_FLOOR_NT` nt), which shows the sampler is right for that state, and (2) Open
does not exceed the matrix mean k (r <= k always).

ON = 100 x Open(switch + trigger) / L and OFF = 100 x Open(switch alone) / L, both in % of the stem.
Designs are ranked by ON - OFF (percentage points), largest first; ties are broken by the CSV order.
The ratio ON / max(OFF, `OFF_FLOOR_PCT`) is listed for information and orders nothing.

In [ ]:
def _state_ok(state, stem_len):
    run, k, m = state["run_nt_sampled"], state["open_nt_sampled"], state["open_nt_matrix"]
    if run is None or k is None:
        return False
    # standard error of the sampled mean k; sd(k) <= L/2 is used when the sd was not saved
    sd = state.get("open_nt_sd", stem_len / 2.0)
    tol = VERIFY_SIGMA * sd / math.sqrt(state["n_drawn"]) + VERIFY_FLOOR_NT
    return abs(k - m) <= tol and run <= m + tol


scored = []
for _d in designs:
    _r = results[_d["rank"]]
    _L = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
    scored.append({
        "design": _d, "csv_rank": int(_d["rank"]), "stem_len": _L,
        "open_t": _r["on"]["run_nt_sampled"], "k_t": _r["on"]["open_nt_sampled"],
        "k_t_matrix": _r["on"]["open_nt_matrix"], "ok_t": _state_ok(_r["on"], _L),
        "k_c_matrix": max((c["open_nt_matrix"] for c in _r.get("controls", [])), default=None),
        "n_controls": len(_r.get("controls", [])),
        "open_a": _r["alone"]["run_nt_sampled"], "k_a": _r["alone"]["open_nt_sampled"],
        "k_a_matrix": _r["alone"]["open_nt_matrix"], "ok_a": _state_ok(_r["alone"], _L),
    })

# ON and OFF as % of the stem; a missing Open makes them (and the difference) None, never 0
for _r in scored:
    _ok = _r["open_t"] is not None and _r["open_a"] is not None
    _r["on_pct"] = 100.0 * _r["open_t"] / _r["stem_len"] if _r["open_t"] is not None else None
    _r["off_alone_pct"] = 100.0 * _r["open_a"] / _r["stem_len"] if _r["open_a"] is not None else None
    # control: exact matrix E[k] with a mismatched trigger is an upper bound of its Open (the sampler is
    # unreliable for mismatched states); the worst (largest) of the controls
    _r["ctrl_pct"] = 100.0 * _r["k_c_matrix"] / _r["stem_len"] if _r["k_c_matrix"] is not None else None
    # OFF = the worst (largest) of the alone state and the controls
    _r["off_pct"] = (max(_r["off_alone_pct"], _r["ctrl_pct"] or 0.0)
                     if _r["off_alone_pct"] is not None else None)
    _r["off_source"] = "control" if (_r["ctrl_pct"] or 0.0) > (_r["off_alone_pct"] or 0.0) else "alone"
    _r["diff_pp"] = _r["on_pct"] - _r["off_pct"] if _ok else None
    _r["ratio"] = _r["on_pct"] / max(_r["off_pct"], OFF_FLOOR_PCT) if _ok else None
    _r["ratio_floored"] = _ok and _r["off_pct"] < OFF_FLOOR_PCT
    # dG: G(switch + trigger) - G(switch alone) - G(trigger alone), ensemble free energies (kcal/mol), exact
    _d = _r["design"]
    _r["dg"] = (folder.partition(f"{_d['switch']}&{_d['trigger']}") - folder.partition(_d["switch"])
                - folder.partition(_d["trigger"]))


# rank by ON - OFF, largest first; a design whose difference is None sorts last
scored.sort(key=lambda r: (r["diff_pp"] is None, -(r["diff_pp"] or 0.0), r["csv_rank"]))
for _pos, _r in enumerate(scored, start=1):
    _r["our_rank"] = _pos

# seed-to-seed noise of ON - OFF: the same designs sampled again with another seed
for _r in scored:
    _c = check_results[_r["design"]["rank"]]
    _on2, _off2 = _c["on"]["run_nt_sampled"], _c["alone"]["run_nt_sampled"]
    _r["diff_pp_check"] = (100.0 * _on2 / _r["stem_len"] - max(100.0 * _off2 / _r["stem_len"], _r["ctrl_pct"] or 0.0)
                          if _on2 is not None and _off2 is not None else None)
_noise = sorted(abs(r["diff_pp"] - r["diff_pp_check"]) for r in scored
                if r["diff_pp"] is not None and r["diff_pp_check"] is not None)
noise_median = statistics.median(_noise)
noise_p95 = _noise[int(0.95 * (len(_noise) - 1))]
n_tied_with_best = sum(1 for r in scored
                       if r["diff_pp"] is not None and scored[0]["diff_pp"] - r["diff_pp"] <= noise_p95)
# near-ties: designs within the noise margin of the best are one tied group, ordered by dG
# (more negative first); the rest follow by ON - OFF
_tied = [r for r in scored if r["diff_pp"] is not None and scored[0]["diff_pp"] - r["diff_pp"] <= noise_p95]
_rest = [r for r in scored if r not in _tied]
_tied.sort(key=lambda r: (r["dg"], r["csv_rank"]))
scored = _tied + _rest
for _pos, _r in enumerate(scored, start=1):
    _r["our_rank"] = _pos
    _r["tied"] = _r in _tied
print(f"seed-to-seed |change of ON - OFF|: median {noise_median:.2f}, 95th percentile {noise_p95:.2f} pp; "
      f"{n_tied_with_best} designs within that of rank 1")

n_ok_t = sum(r["ok_t"] for r in scored)
n_ok_a = sum(r["ok_a"] for r in scored)
print(f"{len(scored)} designs; verified: switch + trigger {n_ok_t}, switch alone {n_ok_a}")
print("top 5 (our rank, CSV rank, ON - OFF):",
      [(r["our_rank"], r["csv_rank"], round(r["diff_pp"], 1)) for r in scored[:5]])

## Correlation with the CSV rank

Spearman rho is the Pearson correlation of the two rank vectors; Kendall tau-b is
(C - D) / sqrt((n0 - n1)(n0 - n2)) over all pairs. Both are +1 when the orders agree.

In [ ]:
def _average_ranks(values):
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2 + 1
        i = j + 1
    return ranks


def spearman_rho(x, y):
    rx, ry = _average_ranks(x), _average_ranks(y)
    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)
    cov = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    vx, vy = sum((a - mx) ** 2 for a in rx), sum((b - my) ** 2 for b in ry)
    return cov / math.sqrt(vx * vy) if vx and vy else None


def kendall_tau_b(x, y):
    n = len(x)
    concordant = discordant = ties_x = ties_y = 0
    for i in range(n):
        for j in range(i + 1, n):
            dx, dy = x[i] - x[j], y[i] - y[j]
            if dx == 0 and dy == 0:
                continue
            if dx == 0:
                ties_x += 1
            elif dy == 0:
                ties_y += 1
            elif (dx > 0) == (dy > 0):
                concordant += 1
            else:
                discordant += 1
    denom = math.sqrt((concordant + discordant + ties_x) * (concordant + discordant + ties_y))
    return (concordant - discordant) / denom if denom else None


_ranked = [r for r in scored if r["diff_pp"] is not None]
csv_ranks = [r["csv_rank"] for r in _ranked]
our_ranks = [r["our_rank"] for r in _ranked]
rho = spearman_rho(csv_ranks, our_ranks)
tau = kendall_tau_b(csv_ranks, our_ranks)
print(f"Spearman rho = {rho:.3f}   Kendall tau-b = {tau:.3f}   over {len(_ranked)} designs")

In [ ]:
import base64, io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm"})
fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.scatter(csv_ranks, our_ranks, s=14, c="#7f8c8d", alpha=0.7, label="all designs")
_top = [r for r in _ranked if r["our_rank"] <= REPORT_TOP_N]
ax.scatter([r["csv_rank"] for r in _top], [r["our_rank"] for r in _top], s=22, c="#c0392b",
           label=f"top {REPORT_TOP_N} (listed in the report)")
ax.plot([1, len(_ranked)], [1, len(_ranked)], "k--", lw=0.8, label="same rank in both")
ax.set_xlabel("rank in the CSV (1 = best)")
ax.set_ylabel("rank by ON - OFF, this report (1 = best)")
ax.invert_yaxis()
ax.set_title(f"Spearman rho = {rho:.2f},  Kendall tau-b = {tau:.2f}")
ax.legend(fontsize=8, loc="lower left")
ax.grid(alpha=0.3)
fig.tight_layout()
_buf = io.BytesIO()
fig.savefig(_buf, format="png", dpi=130)
plt.close(fig)
_corr_png_path = _report_root / f"eu_and_v3_correlation_{datetime.datetime.now():%Y%m%dT%H%M%S}.png"
_corr_png_path.write_bytes(_buf.getvalue())
print(f"wrote correlation plot (outside the report) -> {_corr_png_path}")

## Details for each listed design

For each design in the report table: its parts (domains of the switch and the trigger), and, for the switch
alone and for the switch with its trigger, a 2D picture of the minimum-free-energy structure and the
base-pair-probability matrix. The structure is the single most stable one, drawn from `FoldEngine.mfe`;
the matrix is the exact `FoldEngine.base_pair_probabilities`. Opened by the Details button of the HTML report.

In [ ]:
from _report_figures import DOMAIN_COLORS, bpp_heatmap_png, domain_fill, structure_svg


def _fmt_nt(x):
    return "n/a" if x is None else f"{x:.2f}"


_DOMAIN_ORDER = ("prefix", "toehold", "stem_up", "loop", "stem_down", "pre_kozak", "kozak")


def _switch_line_html(design):
    # the whole switch sequence 5' to 3' through the Kozak, each part shaded in its own colour
    kozak_end = design["domains"]["kozak"][1]
    pieces, legend = [], []
    for name in _DOMAIN_ORDER:
        start, end = design["domains"].get(name, (0, 0))
        if end > start:
            colour = DOMAIN_COLORS.get(name, "#C7D0CA")
            pieces.append(f"<span class='dom' style='background:{colour}55' title='{name}'>"
                          f"{design['switch'][start:end]}</span>")
            legend.append(f"<span class='sw' style='background:{colour}'></span>{name}")
    return (f"<p class='note'>Switch sequence, 5&prime; to 3&prime; through the Kozak ({kozak_end} nt):</p>"
            f"<p class='seqline'>{''.join(pieces)}</p><p class='note'>{' &ensp; '.join(legend)}</p>"
            f"<p class='note'>Trigger ({len(design['trigger'])} nt): <span class='seqline'>{design['trigger']}</span></p>")


def _domain_table_html(design):
    switch, trigger = design["switch"], design["trigger"]
    rows = []
    for name in _DOMAIN_ORDER:
        start, end = design["domains"].get(name, (0, 0))
        if end > start:
            rows.append(
                f"<tr><td>{name}</td><td class='n'>{start + 1}&ndash;{end}</td>"
                f"<td class='n'>{end - start}</td><td class='seq'>{switch[start:end]}</td></tr>"
            )
    rows.append(f"<tr><td>trigger</td><td class='n'>&ndash;</td><td class='n'>{len(trigger)}</td>"
                f"<td class='seq'>{trigger}</td></tr>")
    return ("<table class='booktabs small'><tr><th>part</th><th class='n'>positions (1-based)</th>"
            "<th class='n'>length (nt)</th><th>sequence (5&prime; to 3&prime;)</th></tr>"
            + "".join(rows) + "</table>")


def _state_figures(design, strands, heading):
    # (2D structure SVG, base-pair-probability PNG as base64) of one state
    switch = design["switch"]
    sequence = strands.replace("&", "")
    structure = folder.mfe(strands).structure.replace("&", "")
    kozak_end = design["domains"]["kozak"][1] if design["domains"].get("kozak") else len(switch)
    fills = [f for f in domain_fill(design["domains"]) if f[0] < kozak_end]
    svg = structure_svg(
        folder.layout_coordinates(structure), sequence, structure,
        f"{heading} (minimum-free-energy structure)", domain_fill=fills,
        visible_len=kozak_end, construct_len=len(switch),
    )
    boundary = len(switch) if "&" in strands else None
    png = bpp_heatmap_png(sequence, folder.base_pair_probabilities(strands),
                          f"{heading}: base-pair probabilities", boundary)
    return svg, png


for _r in scored[:REPORT_TOP_N]:
    _d = _r["design"]
    _L = _r["stem_len"]
    _on_text = (f"Open = {_fmt_nt(_r['open_t'])} nt; E[k] = {_fmt_nt(_r['k_t'])} nt "
                f"(matrix {_fmt_nt(_r['k_t_matrix'])}); L = {_L}")
    _ctrl_text = (f"worst E[k] from the matrix over {_r['n_controls']} mismatched triggers = {_fmt_nt(_r['k_c_matrix'])} nt "
                  f"({_r['ctrl_pct']:.1f}% of L; an upper bound of Open); OFF = larger of this and Open(alone) = {_r['off_pct']:.1f}% ({_r['off_source']}); "
                  f"mismatched triggers = triggers of CSV rows {', '.join(str(x) for x in _d.get('control_ranks', []))}; L = {_L}")
    _off_text = (f"Open = {_fmt_nt(_r['open_a'])} nt; E[k] = {_fmt_nt(_r['k_a'])} nt "
                 f"(matrix {_fmt_nt(_r['k_a_matrix'])}); L = {_L}")
    _svg_off, _png_off = _state_figures(_d, _d["switch"], "OFF: switch alone")
    _svg_on, _png_on = _state_figures(_d, _d["switch"] + "&" + _d["trigger"], "ON: switch + trigger")
    _r["detail_html"] = (
        f"<div class='panel' id='p{_r['our_rank']}'><h3>Design rank {_r['our_rank']} "
        f"&middot; stem L = {_L} nt "
        f"<a class='back' href='#results'>&uarr; back to the table</a></h3>"
        f"{_switch_line_html(_d)}{_domain_table_html(_d)}"
        f"<div class='state'><h4>OFF: switch alone</h4><p class='note'>{_off_text}</p>{_svg_off}</div>"
        f"<div class='state'><h4>ON: switch + trigger</h4><p class='note'>{_on_text}</p>{_svg_on}</div>"
        f"<p class='note'><b>Controls: switch + mismatched triggers.</b> {_ctrl_text}</p>"
        f"<div class='bppRow'>"
        f"<figure><img src='data:image/png;base64,{_png_off}'>"
        f"<figcaption>OFF: switch alone</figcaption></figure>"
        f"<figure><img src='data:image/png;base64,{_png_on}'>"
        f"<figcaption>ON: switch + trigger</figcaption></figure>"
        f"</div></div>"
    )
print(f"details built for {REPORT_TOP_N} designs")

## Report

In [ ]:
def _fmt_nt(x):
    return "n/a" if x is None else f"{x:.2f}"


def _fmt_nt_pct(x, length):
    return "n/a" if x is None else f"{x:.2f}&nbsp;({100.0 * x / length:.0f}%)"


_CSS = r'''
@import url("https://cdn.jsdelivr.net/gh/aaaakshat/cm-web-fonts@latest/fonts.css");
@page { size: A4; margin: 20mm 18mm; }
body { font-family: "Computer Modern Serif", "CMU Serif", "Latin Modern Roman", "Times New Roman", serif;
       font-size: 11.5pt; line-height: 1.5; color: #111; max-width: 174mm; margin: 0 auto; }
h1 { font-size: 20pt; font-weight: normal; text-align: center; margin: 0 0 4pt 0; line-height: 1.25; }
.byline { text-align: center; font-size: 10pt; color: #444; margin-bottom: 18pt; }
h2 { font-size: 13.5pt; margin: 22pt 0 6pt 0; font-weight: bold; break-after: avoid; }
p { margin: 6pt 0; text-align: justify; }
dl { margin: 6pt 0; } dt { break-after: avoid; font-weight: bold; float: left; clear: left; width: 34mm; }
dd { break-inside: avoid; margin: 0 0 4pt 36mm; }
code { font-family: "Courier New", monospace; font-size: 9.5pt; }
table { border-collapse: collapse; margin: 8pt auto; font-size: 6.8pt; }
table.booktabs { border-top: 1.5pt solid #111; border-bottom: 1.5pt solid #111; }
table.booktabs th { border-bottom: 0.75pt solid #111; font-weight: bold; padding: 3pt 2pt; text-align: left; }
table.booktabs th.n { white-space: nowrap; }
table.booktabs td { padding: 2pt 2pt; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; }
.ok { color: #1e7e34; } .no { color: #b02a37; font-weight: bold; }
figure { margin: 10pt 0; text-align: center; break-inside: avoid; }
figcaption { font-size: 10pt; color: #333; margin-top: 4pt; text-align: left; }
.note { font-size: 10pt; color: #333; }
.katex { font-size: 1.0em !important; } .katex-display { margin: 0.55em 0 !important; }
th.grp { text-align: center; border-bottom: 0.75pt solid #111; }
th.on { background: #fbe9e6; } th.off { background: #e6eef7; } th.ctl { background: #eef3e6; }
td.okk { background: #e3f3e6; } td.badk { background: #fbe3e0; }
tr.detail > td { padding: 8pt 4pt 14pt 4pt; border-bottom: 0.75pt solid #999; }
.panel h3 { font-size: 12pt; margin: 0 0 6pt 0; } .state h4 { font-size: 11pt; margin: 6pt 0 2pt 0; }
.state { margin-top: 10pt; }
.bppRow { display: flex; gap: 12pt; justify-content: center; margin-top: 10pt; break-inside: avoid; }
.bppRow figure { flex: 1; margin: 2pt; text-align: center; } .bppRow img { max-width: 100%; }
td.seq { font-family: "Courier New", monospace; font-size: 8.5pt; word-break: break-all; }
table.small { font-size: 8.5pt; margin: 4pt 0; }
@media print { .detail, .detail-btn, .noprint { display: none !important; } }
.detail-link, .back, .detail-btn { white-space: nowrap; font-family: inherit; font-size: 9.5pt; color: #1a5fb4;
  text-decoration: underline; font-weight: normal; border: none; background: none; padding: 0; cursor: pointer; }
.back { margin-left: 14pt; }
.seqline { font-family: "Courier New", monospace; font-size: 9pt; word-break: break-all; line-height: 1.7; }
.dom { padding: 1pt 0; } .sw { display: inline-block; width: 8pt; height: 8pt; margin-right: 3pt; }
.abstract { border-left: 3pt solid #1F6FEB; padding: 2pt 8pt; margin: 8pt 0 12pt 0; font-size: 11pt; background: #f4f7fb; }
.appendix .panel { break-before: page; } .appendix .panel:first-of-type { break-before: auto; } h2 + table { break-inside: avoid; } table.small, table.small tr { break-inside: avoid; } #results + p { break-after: avoid; break-inside: avoid; } .appendix .state { break-inside: avoid; }
.appendix svg { max-height: 112mm !important; }
'''

_TEMPLATE = r'''<!doctype html><html><head><meta charset="utf-8"><title>eu_and_v3 report</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/contrib/auto-render.min.js"></script>
<script>document.addEventListener("DOMContentLoaded", function () {
  renderMathInElement(document.body, {delimiters: [
    {left: "\\[", right: "\\]", display: true}, {left: "\\(", right: "\\)", display: false}]});
});</script>
<script>function toggleDetail(id){var r=document.getElementById(id);r.hidden=!r.hidden;}</script>
<style>@@CSS@@</style></head><body>

<h1>Single-switch screen</h1>
<div class="byline">@@SOURCE@@<br>First @@N_DESIGNS@@ designs (the table lists the best @@REPORT_TOP_N@@), folded at
@@TEMPERATURE@@&nbsp;&deg;C with ViennaRNA @@VRNA@@ &middot; @@DATE@@</div>

<div class="abstract"><b>Summary.</b> @@ABSTRACT@@</div>

<h2>1&ensp;Definitions</h2>
<dl>
<dt>switch</dt><dd>the RNA switch strand of a design. It contains one hairpin.</dd>
<dt>trigger</dt><dd>the design's real trigger RNA.</dd>
<dt>stem</dt><dd>the hairpin stem has two sides of \(L\) nucleotides each, \(\mathrm{stem}_{\uparrow}\) (5&prime; side)
and \(\mathrm{stem}_{\downarrow}\) (3&prime; side). In the closed hairpin they pair with each other.
Here \(L\) = @@STEM_LENS@@.</dd>
<dt>state</dt><dd>a set of strands folded together: <i>switch + trigger</i> (the switch with its real trigger),
<i>switch alone</i> (no trigger) or a <i>control</i> (the switch with a mismatched trigger).</dd>
<dt>mismatched trigger</dt><dd>the real trigger of another design of the same table (or, if none qualifies, another sequence offered for this purpose) that shares at most @@CONTROL_OVERLAP@@ nt of identical sequence with the true trigger; @@N_CONTROLS@@ of them per design are used as wrong inputs.</dd>
<dt>prefix, Kozak, payload</dt><dd>the switch RNA starts with a short <i>prefix</i>, then the toehold and the hairpin; the <i>Kozak</i> sequence follows the
hairpin and the <i>payload</i> (the coding sequence, 67 nt here) follows the Kozak. The payload is folded together with the rest but is not drawn.</dd>
<dt>how to read the figures</dt><dd>2D structures: every circle is a nucleotide coloured by part; a <b>gold ring</b> marks a nucleotide paired into the payload
(the undrawn &ldquo;hidden tail&rdquo; after the Kozak) and a <b>pink ring</b> one paired to the (undrawn) trigger strand. Heat maps: the axes run over the
whole folded RNA, including the payload and, for the switch + trigger state, the trigger; the dotted red lines mark where one strand ends and the next begins.</dd>
<dt>toehold, hairpin, loop</dt><dd>the <i>toehold</i> is the single-stranded stretch of the switch where the trigger first binds; the <i>hairpin</i> is the stem (\(\mathrm{stem}_{\uparrow}\) paired with \(\mathrm{stem}_{\downarrow}\)) closed by a <i>loop</i> of unpaired nucleotides. Binding the trigger is meant to open the hairpin.</dd>
<dt>pp</dt><dd>percentage points, the unit of a difference of two percentages.</dd>
<dt>structure</dt><dd>one folded arrangement \(s\) of the strands, i.e. a set of base pairs.</dd>
<dt>\(E(s)\), \(G\)</dt><dd>the energy of structure \(s\), and the free energy of the whole ensemble of
structures of the state (both in kcal/mol).</dd>
<dt>\(\Delta G\)</dt><dd>\(G_{\text{switch + trigger}}-G_{\text{switch alone}}-G_{\text{trigger alone}}\), in kcal/mol, from the exact partition
function: the free energy gained by forming the complex from the two separate molecules. More negative means the trigger binds the switch more strongly (no concentration term). It is listed for information and orders only the tied group; a more stable hairpin makes it less negative, so the ordering slightly favours less stable switches. Strand concentration is not included (the duplex-initiation term of ViennaRNA's dimer energy is).</dd>
<dt>\(P(s)\)</dt><dd>the probability of \(s\) in the ensemble, \(P(s)=\exp\!\big(-(E(s)-G)/RT\big)\), with \(R\) the gas constant and \(T\) the absolute temperature.</dd>
<dt>open nucleotide</dt><dd>a \(\mathrm{stem}_{\uparrow}\) nucleotide that is <i>not</i> paired to a
\(\mathrm{stem}_{\downarrow}\) nucleotide in \(s\) (any one, including a shifted partner). It may be unpaired, or paired to the trigger or to anything else.</dd>
<dt>\(k(s)\)</dt><dd>the number of open nucleotides in \(s\), from \(0\) to \(L\), anywhere on the stem.</dd>
<dt>\(r(s)\)</dt><dd>the length of the longest stretch of <i>consecutive</i> open nucleotides in \(s\),
from \(0\) to \(L\). Always \(r(s)\le k(s)\). Nucleotides in one stretch are open at the same time.</dd>
</dl>

<h2>2&ensp;Open</h2>
<p>For a state, <b>Open</b> is the expected length of the <b>longest run of consecutive open nucleotides</b>
(not the total number of open nucleotides),
\[ \mathrm{Open}\;=\;\mathbb{E}[r]\;=\;\sum_{s}P(s)\,r(s), \]
reported in nucleotides and as a percentage \(100\cdot\mathrm{Open}/L\) of the stem. It is computed for two states:
\[ \begin{aligned}
\mathrm{Open}_{\text{trigger}}&=\mathbb{E}[r]\ \text{of the switch with its real trigger},\\
\mathrm{Open}_{\text{alone}}&=\mathbb{E}[r]\ \text{of the switch alone}.
\end{aligned} \]
A good switch has \(\mathrm{Open}_{\text{trigger}}\) large (the hairpin opens when its trigger is present) and
\(\mathrm{Open}_{\text{alone}}\) small (it stays closed without it).
</p>
<p>Both are expressed as a percentage of the stem and compared,
\[ \begin{aligned}
&\mathrm{ON}=100\,\mathrm{Open}_{\text{trigger}}/L,\qquad \mathrm{OFF}=100\,\max\big(\mathrm{Open}_{\text{alone}},\,\mathrm{Control}\big)/L,\\[1pt]
&\mathrm{ON}-\mathrm{OFF}\ \text{(percentage points)},\qquad \mathrm{ratio}=\mathrm{ON}\,/\max(\mathrm{OFF},\,@@FLOOR@@).
\end{aligned} \]
OFF is the <b>worst</b> (largest) of the switch alone and the controls (below). <b>The designs are ranked by \(\mathrm{ON}-\mathrm{OFF}\)</b> in percentage points, largest first, with
near-ties handled as described below. A percentage of the stem makes designs with different stem lengths comparable (a count of
nucleotides would favour long stems for ON. For OFF the effect may reverse (fraying at the stem ends is roughly constant in nucleotides), so stem lengths are not fully neutral. Counts would favour long stems, which have more nucleotides that can open). The ratio is listed for
information and orders nothing; the floor of @@FLOOR@@% on OFF keeps it finite when OFF is close to zero, and a
ratio computed with the floor is marked with an asterisk.</p>
<p><b>Controls.</b> To see whether Open responds only to the <i>right</i> trigger, every design is also folded with @@N_CONTROLS@@ mismatched triggers. For each, the exact \(\mathbb{E}[k]_{\text{matrix}}\) of the switch with that trigger is an <b>upper bound</b> on its Open (\(\mathrm{Open}\le\mathbb{E}[k]\)); the matrix is used instead of sampling because the sampler is unreliable for such mismatched states. The <i>Control</i> is the largest of these bounds, \(100\cdot\max\mathbb{E}[k]_{\text{matrix}}/L\), and OFF is the larger of it and the alone value, so a switch that any mismatched trigger opens is penalised. The Control column shows it; ON &minus; OFF therefore already contains it.</p>
<p><b>Ties.</b> Repeating the sampling with another seed changes ON &minus; OFF by a small amount (section 5). Designs whose ON &minus; OFF lies within the
95th-percentile of that change from the best one cannot be told apart from it; they form one <i>tied group</i> (marked &dagger; in the table), ranked inside the group by \(\Delta G\) (more negative first, i.e. stronger binding of the trigger) and
listed before the other designs. Only Monte-Carlo noise is counted here; the error of the energy model is larger, so designs just outside the group are not clearly worse either, and tied designs may share a trigger or toehold, so they are not independent candidates. The tied group is the report's shortlist; &dagger; marks it.</p>
<p><b>What Open does and does not measure.</b> The ON state is the ensemble of the switch <i>with its trigger already bound as a complex</i>; the trigger concentration, the binding kinetics and the fraction of switches that actually form the complex are not modelled. Open measures how far the hairpin stem is opened at equilibrium (the exact ensemble of ViennaRNA at the stated
temperature). It is a structural proxy for the hairpin releasing the region behind it; it is not a measurement of translation, of ribosome binding or
start-codon accessibility, of kinetics, or of behaviour in a cell, and trigger accessibility and self-structure, cross-talk between triggers, the openness of \(\mathrm{stem}_{\downarrow}\) and of the start codon, temperature or parameter-set sensitivity, and the origin of the CSV are not assessed; the only control is one mismatched trigger per design (no scrambled triggers or known reference switch), so the scale of the numbers is not calibrated against experiment.</p>
<p>Next to each Open the report lists \(\mathbb{E}[k]\), the <b>expected number of open nucleotides anywhere on
the stem</b>, whether or not they are consecutive,
\[ \mathbb{E}[k]\;=\;\sum_{s}P(s)\,k(s)\;\ge\;\mathrm{Open}. \]
For the listed designs the two nearly coincide (the open nucleotides are mostly contiguous). It is the more relevant number for a state that should stay closed: a stem that is open in scattered places has a small
Open but may have a larger \(\mathbb{E}[k]\). It is given sampled and from the exact matrix. The stem length
\(L\) of each design is listed too, because Open and \(\mathbb{E}[k]\) are counts out of \(L\).</p>

<h2>3&ensp;Estimation and verification</h2>
<p><i>In plain words:</i> Open is estimated by drawing random structures with the probability they have at equilibrium. The exact base-pair probability matrix cannot give Open but can give the average number of open nucleotides, so it is used to check that the drawn structures behave correctly.</p>
<p><b>Sampled estimate.</b> The sum over all structures cannot be written out, so \(N\) structures
\(S_1,\dots,S_N\) are drawn from the ensemble (ViennaRNA stochastic backtracking,
<code>FoldEngine.sample_structures</code>) and averaged,
\[ \mathbb{E}[r]\;\approx\;\frac1N\sum_{i=1}^{N}r(S_i),\qquad N=@@N_SAMPLES@@ . \]
A structure is drawn with probability \(P(s)\), so this estimates the full sum, including structures that are
never drawn. <b>No normalisation is needed:</b> the draws are already distributed according to \(P(s)\), and the average
needs no division by the share of the ensemble that the distinct sampled structures cover (that share is small, since
almost every draw is a different structure; dividing by it would only be needed if the exact \(P(s)\) were summed over
the distinct sampled structures instead). Its statistical error is \(\mathrm{sd}(r)/\sqrt N\).</p>
<p><b>Matrix check.</b> The base-pair probability matrix \(p_{ij}\) (the probability that nucleotides \(i\) and \(j\)
are paired, summed over all structures) gives the expected <i>number</i> of open nucleotides exactly,
\[ \mathbb{E}[k]\;=\;\sum_{i\in \mathrm{stem}_{\uparrow}}\Big(1-\sum_{j\in \mathrm{stem}_{\downarrow}}p_{ij}\Big). \]
It cannot give \(\mathbb{E}[r]\), because it does not know which positions are open together. It does give
the bound \(\mathbb{E}[r]\le\mathbb{E}[k]\). A state is therefore <i>verified</i> (its \(\mathbb{E}[k]\) cells are shaded light green in the table) when
(1) the sampled \(\mathbb{E}[k]\) agrees with the matrix value to within \(\tau\), which shows the sampler is
right for that state, and (2) \(\mathrm{Open}\le\mathbb{E}[k]_{\text{matrix}}+\tau\) (a sanity bound: \(r\le k\) holds for every draw, so this only catches a broken estimate), with \(\tau\) given below. Green therefore shows that the sampler
reproduces the per-nucleotide openness of that state to within \(\tau\); it does not prove the longest runs themselves, which the matrix cannot see.</p>
<p><b>The tolerance.</b> The sampled mean of \(k\) has a standard error \(\mathrm{sd}(k)/\sqrt N\), where \(\mathrm{sd}(k)\) is the standard deviation of \(k\) over the \(N\) draws of that very state. The tolerance is therefore set per state,
\(\tau=@@VERIFY_SIGMA@@\,\mathrm{sd}(k)/\sqrt N+@@VERIFY_FLOOR@@\) nucleotides, so a state that should stay closed (small \(\mathbb{E}[k]\), small \(\mathrm{sd}\)) is checked tightly and a variable one is checked loosely. The constant @@VERIFY_FLOOR@@ nt only
allows for the rounding of the matrix. Verification catches gross sampler faults; \(\tau\) can be as large as the seed-to-seed noise of the ranking, so a green cell does not guarantee ranking-level accuracy. The tolerance sets the shading only, not any Open or \(\mathbb{E}[k]\) value.</p>
<p><b>Strand order.</b> For the switch + trigger state the trigger strand is written first in the sampler (the order that matched the matrix); the switch alone is a single strand.</p>
<p><b>Unverified states.</b> States that fail the check stay in all medians, noise figures and rank correlations; they are listed in section 5. The sampler's dependence on strand order is empirical, not explained.</p>
<p><b>Limitation.</b> ViennaRNA's sampler was found to return wrong draws for some multi-strand states, depending on which strand is written first
(structures whose energy is far above the ensemble's, always with the stem open, so Open and \(\mathbb{E}[k]\) are biased upwards). The strand order is chosen to avoid this, and no draw is filtered or
corrected. In this run @@N_OK_T@@ of @@N_SCORED@@ switch + trigger states and @@N_OK_A@@ of @@N_SCORED@@ switch-alone states passed the check; a state shaded light red should not be trusted. @@RED_TEXT@@</p>

<h2>4&ensp;Parameters</h2>
<table class="booktabs"><tr><th>name</th><th>value</th><th>meaning</th></tr>
@@PARAMS@@
</table>

<h2>5&ensp;Summary</h2>
<table class="booktabs">
<tr><td>designs scored</td><td class="n">@@N_SCORED@@</td></tr>
<tr><td>switch + trigger verified against the matrix</td><td class="n">@@N_OK_T@@ (@@PCT_OK_T@@%)</td></tr>
<tr><td>switch alone verified against the matrix</td><td class="n">@@N_OK_A@@ (@@PCT_OK_A@@%)</td></tr>
<tr><td>median ON / OFF (% of the stem)</td><td class="n">@@MED_T@@% / @@MED_A@@%</td></tr>
<tr><td>median ON &minus; OFF (percentage points)</td><td class="n">@@MED_D@@</td></tr>
<tr><td>median control (worst of @@N_CONTROLS@@ mismatched triggers, matrix upper bound), % of the stem</td><td class="n">@@MED_C@@%</td></tr>
<tr><td>seed-to-seed change of ON &minus; OFF, second independent run: median / 95th percentile (pp)</td><td class="n">@@NOISE_MED@@ / @@NOISE_P95@@</td></tr>
</table>
<p>@@BEST_TEXT@@</p>

<h2 id="results">6&ensp;First @@REPORT_TOP_N@@ designs (tied group first, then by \(\mathrm{ON}-\mathrm{OFF}\))</h2>
<p class="note">&dagger; = member of the tied group (section 2); inside it the order is by \(\Delta G\), so rank 1 need not have the highest ON &minus; OFF. Green or red shading only says whether the sampler agrees with the matrix; it is not a quality flag for the design. * after a ratio = OFF was below the floor, so the floor was used. \(L\) is the stem length of the design. Open is the expected <b>longest run</b> of consecutive open
nucleotides, in nucleotides with the percentage of \(L\) in brackets. \(\mathbb{E}[k]\) is the expected number of
open nucleotides anywhere on the stem (not necessarily consecutive), sampled and exact (matrix); both are shaded
light green when the sampled \(\mathbb{E}[k]\) agrees with the matrix and Open does not exceed it (section 3), light red
otherwise. Control = the largest exact \(\mathbb{E}[k]\) of the switch with any of @@N_CONTROLS@@ mismatched triggers, an upper bound of its Open. OFF is the larger of the alone Open and the Control (in %). The last columns are ON &minus; OFF (percentage points of the stem), the ratio ON / OFF and \(\Delta G\) (kcal/mol). In the HTML version the Details button of each row opens the design's parts, a 2D structure and the
base-pair-probability matrix for both states.</p>
<table class="booktabs">
<thead><tr><th rowspan="2" class="n">rank</th><th rowspan="2" class="n">\(L\)</th>
<th colspan="3" class="grp on">ON &ensp;<small>switch + trigger</small></th>
<th colspan="3" class="grp off">OFF &ensp;<small>switch alone</small></th>
<th class="grp ctl">Control &ensp;<small>5 mismatched triggers</small></th>
<th rowspan="2" class="n">ON &minus; OFF<br>(pp)</th><th rowspan="2" class="n">ratio<br>ON / OFF</th><th rowspan="2" class="n">\(\Delta G\)<br>(kcal/mol)<br>lower = tighter</th>@@BTNHEAD@@</tr>
<tr><th class="n">\(\mathrm{Open}_{\text{trigger}}\)<br>longest run</th><th class="n">\(\mathbb{E}[k]\)<br>sampled</th>
<th class="n">\(\mathbb{E}[k]\)<br>matrix</th>
<th class="n">\(\mathrm{Open}_{\text{alone}}\)<br>longest run</th><th class="n">\(\mathbb{E}[k]\)<br>sampled</th>
<th class="n">\(\mathbb{E}[k]\)<br>matrix</th>
<th class="n">\(\mathbb{E}[k]\)<br>matrix (&ge; Open)</th></tr></thead>
@@ROWS@@
</table>
@@APPENDIX@@
</body></html>'''

_stem_text = ", ".join(str(n) for n in sorted({r["stem_len"] for r in scored}))
_param_rows = [
    ("N_DESIGNS", N_DESIGNS, "first rows of the CSV that are scored"),
    ("TEMPERATURE_C", TEMPERATURE_C, "folding temperature, &deg;C"),
    ("N_SAMPLES", N_SAMPLES, "structures drawn from the ensemble for each state"),
    ("SEED", SEED, "random seed for the sampler (a rerun repeats)"),
    ("CHECK_SEED_OFFSET", CHECK_SEED_OFFSET, "a second sampling run uses SEED + this; it only measures the noise (section 5)"),
    ("VERIFY_SIGMA", VERIFY_SIGMA, "&tau;: allowed disagreement with the matrix, in standard errors of the sampled mean"),
    ("VERIFY_FLOOR_NT", VERIFY_FLOOR_NT, "added to &tau; (nt)"),
    ("N_CONTROLS", N_CONTROLS, "mismatched triggers per design; OFF is the worst of the alone state and these"),
    ("CONTROL_MAX_OVERLAP_NT", CONTROL_MAX_OVERLAP_NT, "the mismatched control trigger shares at most this many nt of identical sequence with the true trigger"),
    ("OFF_FLOOR_PCT", OFF_FLOOR_PCT, "OFF is not allowed below this many % of the stem in the ratio ON / OFF"),
    ("REPORT_TOP_N", REPORT_TOP_N, "designs listed in the table of section 6"),
]
_params_html = "".join(
    f"<tr><td><code>{n}</code></td><td>{v}</td><td>{m}</td></tr>" for n, v, m in _param_rows
)
def _fmt_pp(x):
    return "n/a" if x is None else f"{x:.1f}"


def _fmt_ratio(r):
    if r["ratio"] is None:
        return "n/a"
    return f"{r['ratio']:.0f}" + ("*" if r["ratio_floored"] else "")


def _row_html(r, interactive=True):
    L = r["stem_len"]
    cls_t = "okk" if r["ok_t"] else "badk"
    cls_a = "okk" if r["ok_a"] else "badk"
    rid = f"d{r['our_rank']}"
    button = (f"<td class='noprint'><button class='detail-btn' onclick=\"toggleDetail('{rid}')\">"
              "Details</button></td>") if interactive else (
        f"<td><a class='detail-link' href='#p{r['our_rank']}'>Details</a></td>")
    detail_row = (f"<tr class='detail' id='{rid}' hidden><td colspan='13'>{r['detail_html']}</td></tr>"
                  if interactive else "")
    return (
        f"<tr><td class='n'>{r['our_rank']}{'&dagger;' if r['tied'] else ''}</td><td class='n'>{L}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['open_t'], L)}</td>"
        f"<td class='n {cls_t}'>{_fmt_nt(r['k_t'])}</td><td class='n {cls_t}'>{_fmt_nt(r['k_t_matrix'])}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['open_a'], L)}</td>"
        f"<td class='n {cls_a}'>{_fmt_nt(r['k_a'])}</td><td class='n {cls_a}'>{_fmt_nt(r['k_a_matrix'])}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['k_c_matrix'], L)}</td>"
        f"<td class='n'><b>{_fmt_pp(r['diff_pp'])}</b></td><td class='n'>{_fmt_ratio(r)}</td>"
        f"<td class='n'>{r['dg']:.1f}</td>"
        f"{button}</tr>{detail_row}"
    )


_rows_html = "".join(_row_html(r) for r in scored[:REPORT_TOP_N])
_n = len(scored)
_med = lambda key: statistics.median(  # noqa: E731
    100.0 * r[key] / r["stem_len"] for r in scored if r[key] is not None)
_fields = {
    "CSS": _CSS, "SOURCE": html.escape(Path(CSV_PATH).name),
    "ABSTRACT": (f"Each of the first {_n} designs of the CSV (in the order of the input file, whose own ranking is not described here) is folded with and without its trigger. <b>ON</b> is how far the hairpin stem is open with the trigger, <b>OFF</b> how far without it or with a wrong trigger (the worst of the alone state and 5 mismatched triggers), "
        f"both as a percentage of the stem (100% = fully open). A good switch has ON near 100% and OFF near 0%. Here the median ON is {statistics.median(r['on_pct'] for r in scored if r['on_pct'] is not None):.1f}% and the median OFF "
        f"{statistics.median(r['off_pct'] for r in scored if r['off_pct'] is not None):.1f}%. Designs are ranked by ON &minus; OFF; {n_tied_with_best} of {_n} lie within the run-to-run difference (95th percentile of the change of ON &minus; OFF between two independent samplings: {noise_p95:.2f} points) of the best and cannot be told apart by that score. "
        f"They are marked &dagger; and are the shortlist; among them pick by \\(\\Delta G\\) (the order of the table) or by the details. This is a structure-based prediction only, with no experimental calibration."), "N_DESIGNS": _n, "TEMPERATURE": f"{TEMPERATURE_C:g}",
    "DATE": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), "STEM_LENS": _stem_text,
    "N_SAMPLES": N_SAMPLES, "VERIFY_SIGMA": f"{VERIFY_SIGMA:g}", "VERIFY_FLOOR": f"{VERIFY_FLOOR_NT:g}",
    "RED_TEXT": ("All listed designs are shaded green." if all(r["ok_t"] and r["ok_a"] for r in scored[:REPORT_TOP_N])
                 else f"{sum(not (r['ok_t'] and r['ok_a']) for r in scored[:REPORT_TOP_N])} of the listed designs have a red cell."), "PARAMS": _params_html, "N_SCORED": _n,
    "N_OK_T": n_ok_t, "PCT_OK_T": f"{100.0 * n_ok_t / _n:.1f}",
    "N_OK_A": n_ok_a, "PCT_OK_A": f"{100.0 * n_ok_a / _n:.1f}",
    "MED_T": f"{statistics.median(r['on_pct'] for r in scored if r['on_pct'] is not None):.1f}",
    "MED_A": f"{statistics.median(r['off_pct'] for r in scored if r['off_pct'] is not None):.1f}",
    "MED_D": f"{statistics.median(r['diff_pp'] for r in scored if r['diff_pp'] is not None):.1f}",
    "FLOOR": f"{OFF_FLOOR_PCT:g}", "CONTROL_OVERLAP": CONTROL_MAX_OVERLAP_NT, "N_CONTROLS": N_CONTROLS,
    "MED_C": f"{statistics.median(r['ctrl_pct'] for r in scored if r['ctrl_pct'] is not None):.1f}",
    "VRNA": folder.versions().get("ViennaRNA", ""),
    "NOISE_MED": f"{noise_median:.2f}", "NOISE_P95": f"{noise_p95:.2f}",
    "BEST_TEXT": (f"Highest ON &minus; OFF: {max(r['diff_pp'] for r in scored if r['diff_pp'] is not None):.1f} points. "
                  f"{n_tied_with_best} of the {len(scored)} designs lie within the 95th-percentile seed-to-seed change "
                  f"({noise_p95:.2f} points) of it. They form the tied group (&dagger;), which is the shortlist; inside it designs are ordered by \\(\\Delta G\\), not by ON &minus; OFF."),
    "REPORT_TOP_N": REPORT_TOP_N, "ROWS": _rows_html,
}
def _fill(template, fields):
    for _key, _value in fields.items():
        template = template.replace(f"@@{_key}@@", str(_value))
    return template


_btn_head = '<th rowspan="2" class="noprint"></th>'
_appendix = ("<div class='appendix'><h2>7&ensp;Design details</h2><p class='note'>About two pages per design of the "
             "table above (the 2D structures of the switch alone and of the switch with its trigger, then their base-pair probabilities side by side): its parts, and the minimum-free-energy structure and base-pair probabilities of "
             "each state. (The HTML version shows the same under each row's Details button.)</p>"
             + "".join(r["detail_html"] for r in scored[:REPORT_TOP_N]) + "</div>")
# interactive HTML: buttons, details inline. Print version (PDF source): no buttons, details as an appendix.
report_html = _fill(_TEMPLATE, {**_fields, "BTNHEAD": _btn_head, "APPENDIX": ""})
_print_fields = {**_fields, "BTNHEAD": '<th rowspan="2"></th>', "APPENDIX": _appendix,
                 "ROWS": "".join(_row_html(r, interactive=False) for r in scored[:REPORT_TOP_N])}
report_print_html = _fill(_TEMPLATE, _print_fields)

_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
report_html_path = _report_root / f"eu_and_v3_report_{_ts}.html"
report_html_path.write_text(report_html)
_print_html_path = _report_root / f"eu_and_v3_report_{_ts}_print.html"
_print_html_path.write_text(report_print_html)

import csv as _csv
all_csv_path = _report_root / f"eu_and_v3_all_designs_{_ts}.csv"
with all_csv_path.open("w", newline="") as _fh:
    _w = _csv.writer(_fh)
    _w.writerow(["our_rank", "csv_rank", "stem_len", "Open_trigger_nt", "Ek_trigger_sampled_nt",
                 "Ek_trigger_matrix_nt", "trigger_verified", "Open_alone_nt", "Ek_alone_sampled_nt",
                 "Ek_alone_matrix_nt", "alone_verified", "ON_pct", "OFF_pct", "ON_minus_OFF_pp",
                 "ratio_ON_over_OFF", "dG_kcal_per_mol", "tied_group", "OFF_alone_pct",
                 "control_matrix_upper_bound_pct", "OFF_source", "control_trigger_csv_ranks"])
    for r in scored:
        _w.writerow([r["our_rank"], r["csv_rank"], r["stem_len"], r["open_t"], r["k_t"],
                     r["k_t_matrix"], r["ok_t"], r["open_a"], r["k_a"], r["k_a_matrix"], r["ok_a"],
                     r["on_pct"], r["off_pct"], r["diff_pp"], r["ratio"], r["dg"], r["tied"], r["off_alone_pct"], r["ctrl_pct"],
                     r["off_source"], " ".join(str(x) for x in r["design"].get("control_ranks", []))])
print(f"wrote {all_csv_path}")


def _render_pdf(html_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        script = base / "src/engine/gates/notebooks/toehold/render_pdf.py"
        if script.exists():
            break
    else:
        print("render_pdf.py not found -- HTML written, PDF skipped")
        return None
    pdf_path = html_path.with_suffix(".pdf")
    subprocess.run([sys.executable, str(script), str(html_path), str(pdf_path), "--portrait"],
                   check=True)
    print(f"wrote {pdf_path}")
    return pdf_path


_pdf_made = _render_pdf(_print_html_path)   # PDF: table + one page of details per design
report_pdf_path = _pdf_made.rename(report_html_path.with_suffix(".pdf")) if _pdf_made else None
_print_html_path.unlink()   # only the PDF's source
print(f"HTML report with Details buttons -> {report_html_path}")

# Part 2 — AND report

A = 5' half, B = 3' half, fused in all ordered pairs of the top `AND_TOP_N` designs of part 1; four states per candidate (alone, +A, +B, +A+B) and mismatched-trigger controls. Names in this part end in `_and` where they would otherwise clash with part 1.

## AND candidates: fuse the pairs, then sample the four states of each

A candidate is an ordered pair (A = 5' half, B = 3' half) with a spacer variant. Pairs whose two triggers
are too similar are excluded before anything is folded.

In [ ]:
import itertools

from _joint_state_parallel import (
    generate_spacer_candidates,
    pick_fake_triggers,
    pick_optimized_spacer,
    trigger_pair_too_similar,
)
from _sampled_and_parallel import AndJob, run_parallel_and
from _switch_construct import fuse

# The pool is the top AND_TOP_N designs of the single report above (its tied group first, then by
# ON - OFF), taken from memory: nothing is scored or sampled again here.
N_SINGLE = N_DESIGNS
_single_csv = all_csv_path   # the table of the single report, named in the AND report's text
pool = []
for _row in scored[:AND_TOP_N]:
    _d = _row["design"]
    _d["single_rank"] = _row["our_rank"]
    _d["single_diff_pp"] = _row["diff_pp"]
    pool.append(_d)
single_results = {}   # the AND results file keeps the key; the singles are in the single report
_saved_and = json.loads(Path(RESULTS_TO_LOAD_AND).read_text()) if RESULTS_TO_LOAD_AND is not None else None
print(f"pool = top {len(pool)} of the single report: "
      f"{[(d['single_rank'], d['rank']) for d in pool]}")


In [ ]:
SPACER_CANDIDATES = generate_spacer_candidates(
    SPACER_CANDIDATE_POOL_SIZE, SPACER_MIN_LENGTH, SPACER_MAX_LENGTH, seed=SPACER_SEED,
)

_control_pool_and = designs + [{"rank": f"extra{_k}", "trigger": _t} for _k, _t in enumerate(CONTROL_EXTRA_TRIGGERS)]
candidates = {}          # key -> dict(d5, d3, built, spacer_label, spacer_seq)
_and_jobs = []
_n_self = _n_similar = _n_fuse_failed = 0
for _label in SPACER_VARIANTS:
    for _d5, _d3 in itertools.product(pool, pool):
        if _d5["rank"] == _d3["rank"]:
            _n_self += 1
            continue
        if trigger_pair_too_similar(_d5["trigger"], _d3["trigger"],
                                    max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                                    wobble_max_fraction=WOBBLE_MAX_FRACTION):
            _n_similar += 1
            continue
        if _label == "spacer":
            _spacer_seq, _ = pick_optimized_spacer(_d5, _d3, SPACER_CANDIDATES)
        else:
            _spacer_seq = ""
        _built, _problem = fuse(_d5, _d3, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER,
                                spacer=_spacer_seq)
        if _problem is not None:
            _n_fuse_failed += 1
            continue
        _key = f"{_d5['rank']}+{_d3['rank']}|{_label}"
        # controls: N_CONTROLS states, each with two mismatched triggers (real triggers of other designs,
        # at most CONTROL_MAX_OVERLAP_NT nt of identity with either real trigger)
        _real = (_d5["trigger"], _d3["trigger"])
        _fakes = pick_fake_triggers(_real, {_d5["rank"], _d3["rank"]}, _control_pool_and, 2 * N_CONTROLS,
                                    start_offset=len(candidates), max_overlap_nt=CONTROL_MAX_OVERLAP_NT)
        _pairs = tuple((_fakes[_k]["trigger"], _fakes[_k + 1]["trigger"])
                       for _k in range(0, len(_fakes) - 1, 2))
        candidates[_key] = {"d5": _d5, "d3": _d3, "built": _built,
                            "spacer_label": _label, "spacer_seq": _spacer_seq, "controls": _pairs}
        _and_jobs.append(AndJob(
            key=_key, fused=_built["fused"], trigger_a=_d5["trigger"], trigger_b=_d3["trigger"],
            stem_up_a=_built["domains_5p"]["stem_up"], stem_down_a=_built["domains_5p"]["stem_down"],
            stem_up_b=_built["domains_3p"]["stem_up"], stem_down_b=_built["domains_3p"]["stem_down"],
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, seed=SEED,
            controls=candidates[_key]["controls"],
        ))
print(f"{len(candidates)} candidates ({_n_self} self-pairs, {_n_similar} excluded as trigger pairs too "
      f"similar, {_n_fuse_failed} failed fuse())")

if RESULTS_TO_LOAD_AND is None:
    _t0_and = time.time()
    and_results = run_parallel_and(_and_jobs, max_workers=MAX_WORKERS)
    and_check = run_parallel_and(
        [j._replace(seed=SEED + CHECK_SEED_OFFSET) for j in _and_jobs], max_workers=MAX_WORKERS)
    print(f"sampled {len(_and_jobs)} candidates x 4 states twice (second run = noise check) in "
          f"{time.time() - _t0_and:.0f}s")
    _results_path_and = _report_root / f"eu_and_v3_and_results_{datetime.datetime.now():%Y%m%dT%H%M%S}.json"
    _results_path_and.write_text(json.dumps(
        {"singles": single_results, "and": and_results, "and_check": and_check}))
    print(f"wrote {_results_path_and}")
else:
    and_results, and_check = _saved_and["and"], _saved_and["and_check"]
    print(f"loaded results for {len(and_results)} candidates from {RESULTS_TO_LOAD_AND}")

## Open, checks and ranking

Per state the sampling gives E[r_A], E[r_B] (`run_a_sampled`, `run_b_sampled`), E[min(r_A, r_B)]
(`open_sampled`), E[k_A], E[k_B] and E[min(k_A, k_B)], and the matrix gives each hairpin's E[k] exactly.
A state is *verified* when the sampled E[k_A] and E[k_B] each match the matrix within `VERIFY_SIGMA` standard errors (+ `VERIFY_FLOOR_NT` nt) (that
hairpin's and its Open does not exceed the matrix bound (r <= k always).

Open(+A+B) = E[r_A] + E[r_B] and ON = 100 x Open(+A+B) / (L_A + L_B). OFF = 100 x max(Open(alone),
Open(+A), Open(+B)) / L with Open = E[min(r_A, r_B)] and L = min(L_A, L_B). Candidates are ranked by
ON - OFF (percentage points); candidates within the seed-to-seed noise of the best form a tied group ordered by dG.

In [ ]:
_STATES = ("alone", "+A", "+B", "+A+B")
_CHECKED = _STATES


def _open_value(state, name):
    # Open of a state: E[r_A] + E[r_B] for +A+B (ON), E[min(r_A, r_B)] for the others (OFF)
    if state["open_sampled"] is None:
        return None
    if name == "+A+B":
        return state["run_a_sampled"] + state["run_b_sampled"]
    return state["open_sampled"]


def _state_ok_and(state, name, l_a, l_b):
    if state["open_sampled"] is None or state["k_a_sampled"] is None:
        return False
    # per-hairpin tolerance: standard errors of the sampled means (sd(k) <= L/2 if the sd was not saved)
    root_n = math.sqrt(state["n_drawn"])
    tol_a = VERIFY_SIGMA * (state.get("k_a_sd") or l_a / 2.0) / root_n + VERIFY_FLOOR_NT
    tol_b = VERIFY_SIGMA * (state.get("k_b_sd") or l_b / 2.0) / root_n + VERIFY_FLOOR_NT
    if name == "+A+B":   # r_A + r_B <= k_A + k_B, and the matrix gives E[k_A] + E[k_B] exactly
        bound = state["k_a_matrix"] + state["k_b_matrix"] + tol_a + tol_b
    else:                # min(r_A, r_B) <= min(k_A, k_B) <= min(E[k_A], E[k_B]) in expectation
        bound = min(state["k_a_matrix"], state["k_b_matrix"]) + max(tol_a, tol_b)
    return (abs(state["k_a_sampled"] - state["k_a_matrix"]) <= tol_a
            and abs(state["k_b_sampled"] - state["k_b_matrix"]) <= tol_b
            and _open_value(state, name) <= bound)


def _state_strands(c, state):
    fused, ta, tb = c["built"]["fused"], c["d5"]["trigger"], c["d3"]["trigger"]
    return {"alone": fused, "+A": f"{fused}&{ta}", "+B": f"{fused}&{tb}",
            "+A+B": f"{fused}&{tb}&{ta}"}[state]


def _metrics(res, la, lb):
    # ON (sum definition), ON (min definition), OFF, differences and ratio of one candidate's states
    lmin = min(la, lb)
    opens = {s: _open_value(res[s], s) for s in _STATES}
    off_states = {s: opens[s] for s in ("alone", "+A", "+B")}
    # controls: exact matrix bound min(E[k_A], E[k_B]) >= E[min(r_A, r_B)] for each control state (the
    # sampler is unreliable for mismatched triggers); the worst of them enters OFF
    ctrl = [min(c["k_a_matrix"], c["k_b_matrix"]) for c in res.get("controls", [])]
    ctrl_pct = 100.0 * max(ctrl) / lmin if ctrl else None
    if ctrl:
        off_states["control"] = max(ctrl)
    if opens["+A+B"] is None or None in off_states.values():
        return {"ctrl_pct": ctrl_pct, "open": opens, "on_pct": None, "on_min_pct": None, "off_pct": None, "off_source": None,
                "diff_pp": None, "diff_min_pp": None, "ratio": None, "ratio_floored": False}
    off_source = max(off_states, key=off_states.get)
    off_pct = 100.0 * off_states[off_source] / lmin
    on_sum_pct = 100.0 * opens["+A+B"] / (la + lb)
    on_min_pct = 100.0 * res["+A+B"]["open_sampled"] / lmin
    on_pct = on_sum_pct   # ON used for the ranking: E[r_A + r_B] / (L_A + L_B)
    return {"ctrl_pct": ctrl_pct, "open": opens, "on_pct": on_pct, "on_min_pct": on_min_pct, "off_pct": off_pct,
            "off_source": off_source, "diff_pp": on_pct - off_pct, "diff_min_pp": on_min_pct - off_pct,
            "ratio": on_pct / max(off_pct, OFF_FLOOR_PCT), "ratio_floored": off_pct < OFF_FLOOR_PCT}


scored_and = []
for _key, _c in candidates.items():
    _res = and_results[_key]
    _b = _c["built"]
    _la = _b["domains_5p"]["stem_up"][1] - _b["domains_5p"]["stem_up"][0]
    _lb = _b["domains_3p"]["stem_up"][1] - _b["domains_3p"]["stem_up"][0]
    _ok = {s: _state_ok_and(_res[s], s, _la, _lb) for s in _STATES}
    _g = {s: folder.partition(_state_strands(_c, s)) for s in _STATES}
    # dG = G(+A+B) - min over the OFF states of [G(state) + G of the triggers that state lacks, folded alone]
    _gta, _gtb = folder.partition(_c["d5"]["trigger"]), folder.partition(_c["d3"]["trigger"])
    _ref = min(_g["alone"] + _gta + _gtb, _g["+A"] + _gtb, _g["+B"] + _gta)
    _m = _metrics(_res, _la, _lb)
    _m_check = _metrics(and_check[_key], _la, _lb)
    scored_and.append({
        "key": _key, **_c, "res": _res, "ok": _ok, **_m,
        "diff_pp_check": _m_check["diff_pp"],
        "l_a": _la, "l_b": _lb, "l_min": min(_la, _lb),
        "dg": _g["+A+B"] - _ref,
        "n_controls": len(_res.get("controls", [])),
    })

scored_and.sort(key=lambda r: (r["diff_pp"] is None, -(r["diff_pp"] or 0.0), r["key"]))
for _pos_and, _r_and in enumerate(scored_and, start=1):
    _r_and["our_rank"] = _pos_and

# rank by the min definition of ON as well, for comparison
_by_min = sorted(scored_and, key=lambda r: (r["diff_min_pp"] is None, -(r["diff_min_pp"] or 0.0), r["key"]))
for _pos_and, _r_and in enumerate(_by_min, start=1):
    _r_and["rank_min"] = _pos_and


def _spearman(x, y):
    n = len(x)
    mx = (n + 1) / 2
    cov = sum((a - mx) * (b - mx) for a, b in zip(x, y))
    return cov / sum((a - mx) ** 2 for a in x)


rank_agreement = _spearman([r["our_rank"] for r in scored_and], [r["rank_min"] for r in scored_and])
top_overlap = len({r["key"] for r in scored_and[:REPORT_TOP_N]} & {r["key"] for r in _by_min[:REPORT_TOP_N]})

# seed-to-seed noise of ON - OFF: the same candidates sampled again with another seed
_noise_and = sorted(abs(r["diff_pp"] - r["diff_pp_check"]) for r in scored_and
                if r["diff_pp"] is not None and r["diff_pp_check"] is not None)
noise_median_and = statistics.median(_noise_and)
noise_p95_and = _noise_and[int(0.95 * (len(_noise_and) - 1))]
n_tied_with_best_and = sum(1 for r in scored_and
                       if r["diff_pp"] is not None and scored_and[0]["diff_pp"] - r["diff_pp"] <= noise_p95_and)
# near-ties: candidates within the noise margin of the best are one tied group, ordered by dG
_tied_and = [r for r in scored_and if r["diff_pp"] is not None and scored_and[0]["diff_pp"] - r["diff_pp"] <= noise_p95_and]
_rest_and = [r for r in scored_and if r not in _tied_and]
_tied_and.sort(key=lambda r: (r["dg"], r["key"]))
scored_and = _tied_and + _rest_and
for _pos_and, _r_and in enumerate(scored_and, start=1):
    _r_and["our_rank"] = _pos_and
    _r_and["tied"] = _r_and in _tied_and
print(f"seed-to-seed |change of ON - OFF|: median {noise_median_and:.2f}, 95th percentile {noise_p95_and:.2f} pp; "
      f"{n_tied_with_best_and} candidates within that of rank 1")
print(f"ranking by the sum definition (used) vs the min definition: rank correlation {rank_agreement:.3f}, "
      f"top-{REPORT_TOP_N} overlap {top_overlap}")
_unverified = [(r["our_rank"], s) for r in scored_and for s in _STATES if not r["ok"][s]]
n_ok = {s: sum(r["ok"][s] for r in scored_and) for s in _CHECKED}
print(f"{len(scored_and)} candidates; states verified: {n_ok}")
print("top 5:", [(r["our_rank"], r["key"], round(r["diff_pp"], 1)) for r in scored_and[:5]])

## Details for each listed candidate

Parts of the fused construct (hairpin A, optional spacer, hairpin B) and both triggers, the 2D picture of
the minimum-free-energy structure for the OFF state (alone) and the ON state (+A+B), and the exact
base-pair-probability matrix of all four states.

In [ ]:
from _report_figures import (
    DOMAIN_COLORS, DOMAIN_COLORS_B, SPACER_COLOR, bpp_heatmap_png, domain_fill, structure_svg,
)

_HIDDEN = ("exp_gene", "opt_exp_gene")


def _fmt_nt_and(x):
    return "n/a" if x is None else f"{x:.2f}"


def _parts(c):
    # (hairpin, name, start, end, colour) in fused order, payload hidden
    b = c["built"]
    rows = []
    for name, (s, e) in b["domains_5p"].items():
        if e > s and name not in _HIDDEN:
            rows.append(("A", name, s, e, DOMAIN_COLORS.get(name, "#C7D0CA")))
    sp0, sp1 = b["spacer_span"]
    if sp1 > sp0:
        rows.append(("", "spacer", sp0, sp1, SPACER_COLOR))
    for name, (s, e) in b["domains_3p"].items():
        if e > s and name not in _HIDDEN:
            rows.append(("B", name, s, e, DOMAIN_COLORS_B.get(name, "#C7D0CA")))
    return sorted(rows, key=lambda t: t[2])


def _construct_html(c):
    fused = c["built"]["fused"]
    parts = _parts(c)
    visible = max(p[3] for p in parts)
    line = "".join(
        f"<span class='dom' style='background:{col}55' title='{h} {n}'>{fused[s:e]}</span>"
        for h, n, s, e, col in parts
    )
    legend = " &ensp; ".join(
        f"<span class='sw' style='background:{col}'></span>{(n + ' (' + h + ')') if h else n}"
        for h, n, s, e, col in parts
    )
    rows = "".join(
        f"<tr><td>{h or '&ndash;'}</td><td>{n}</td><td class='n'>{s + 1}&ndash;{e}</td>"
        f"<td class='n'>{e - s}</td><td class='seq'>{fused[s:e]}</td></tr>"
        for h, n, s, e, col in parts
    )
    rows += (f"<tr><td>A</td><td>trigger A</td><td class='n'>&ndash;</td>"
             f"<td class='n'>{len(c['d5']['trigger'])}</td><td class='seq'>{c['d5']['trigger']}</td></tr>"
             f"<tr><td>B</td><td>trigger B</td><td class='n'>&ndash;</td>"
             f"<td class='n'>{len(c['d3']['trigger'])}</td><td class='seq'>{c['d3']['trigger']}</td></tr>")
    return (f"<p class='note'>Fused switch, 5&prime; to 3&prime; through the Kozak of hairpin B ({visible} nt); "
            f"hairpin A is the 5&prime; half, hairpin B the 3&prime; half:</p>"
            f"<p class='seqline'>{line}</p><p class='note'>{legend}</p>"
            f"<table class='booktabs small'><tr><th>hairpin</th><th>part</th>"
            f"<th class='n'>positions (1-based)</th><th class='n'>length (nt)</th>"
            f"<th>sequence (5&prime; to 3&prime;)</th></tr>{rows}</table>"), visible


def _state_figures_and(c, state, visible, heading):
    strands = _state_strands(c, state)
    fused = c["built"]["fused"]
    sequence = strands.replace("&", "")
    structure = folder.mfe(strands).structure.replace("&", "")
    b = c["built"]
    fills = (
        domain_fill({n: v for n, v in b["domains_5p"].items() if n not in _HIDDEN}, DOMAIN_COLORS, " (A)")
        + domain_fill({n: v for n, v in b["domains_3p"].items() if n not in _HIDDEN}, DOMAIN_COLORS_B, " (B)")
    )
    sp0, sp1 = b["spacer_span"]
    if sp1 > sp0:
        fills.append((sp0, sp1, SPACER_COLOR, "spacer"))
    fills = [f for f in fills if f[0] < visible]
    svg = structure_svg(
        folder.layout_coordinates(structure), sequence, structure,
        f"{heading} (minimum-free-energy structure)", domain_fill=fills,
        visible_len=visible, construct_len=len(fused),
    )
    return svg


def _bpp_png(c, state, dpi=120):
    strands = _state_strands(c, state)
    fused = c["built"]["fused"]
    parts = strands.split("&")
    junctions = [len(fused)] if len(parts) > 1 else []
    if len(parts) > 2:
        junctions.append(len(fused) + len(parts[1]))
    return bpp_heatmap_png(strands.replace("&", ""), folder.base_pair_probabilities(strands),
                           f"{state}: base-pair probabilities", junctions, dpi=dpi)


def _state_text(c, state):
    r = c["res"][state]
    if state == "+A+B":
        return (f"Open (sum, used for ON) = E[r_A] + E[r_B] = {_fmt_nt_and(r['run_a_sampled'])} + {_fmt_nt_and(r['run_b_sampled'])} = "
                f"{_fmt_nt_and(c['open']['+A+B'])} nt of {c['l_a'] + c['l_b']}; "
                f"Open (min, information) = {_fmt_nt_and(r['open_sampled'])} nt of {c['l_min']}; "
                f"hairpin A: E[k] {_fmt_nt_and(r['k_a_sampled'])} (matrix {_fmt_nt_and(r['k_a_matrix'])}), "
                f"hairpin B: E[k] {_fmt_nt_and(r['k_b_sampled'])} (matrix {_fmt_nt_and(r['k_b_matrix'])})")
    return (f"Open = {_fmt_nt_and(r['open_sampled'])} nt; E[min(k_A, k_B)] = {_fmt_nt_and(r['k_min_sampled'])} nt; "
            f"hairpin A: E[k] {_fmt_nt_and(r['k_a_sampled'])} (matrix {_fmt_nt_and(r['k_a_matrix'])}), "
            f"hairpin B: E[k] {_fmt_nt_and(r['k_b_sampled'])} (matrix {_fmt_nt_and(r['k_b_matrix'])})")


def _detail_panel_html(_r_and, dpi=120):
    _construct, _visible = _construct_html(_r_and)
    _svg_off = _state_figures_and(_r_and, "alone", _visible, "OFF: alone")
    _svg_on = _state_figures_and(_r_and, "+A+B", _visible, "ON: +A+B")
    _pngs = "".join(
        f"<figure><img src='data:image/png;base64,{_bpp_png(_r_and, s, dpi)}'><figcaption>{s}</figcaption></figure>"
        for s in _STATES
    )
    _spacer = f"spacer {len(_r_and['spacer_seq'])} nt" if _r_and["spacer_seq"] else "no spacer"
    return (
        f"<div class='panel' id='p{_r_and['our_rank']}'><h3>Candidate rank {_r_and['our_rank']} &middot; "
        f"A = single rank {_r_and['d5']['single_rank']}, B = single rank {_r_and['d3']['single_rank']} "
        f"&middot; {_spacer} &middot; stems L<sub>A</sub> = {_r_and['l_a']}, L<sub>B</sub> = {_r_and['l_b']} nt "
        f"<a class='back' href='#results'>&uarr; back to the table</a></h3>"
        f"{_construct}"
        f"<div class='state'><h4>OFF: alone</h4><p class='note'>{_state_text(_r_and, 'alone')}</p>{_svg_off}</div>"
        f"<div class='state'><h4>ON: +A+B</h4><p class='note'>{_state_text(_r_and, '+A+B')}</p>{_svg_on}</div>"
        f"<div class='bppRow'>{_pngs}</div>"
        f"<p class='note'>+A: {_state_text(_r_and, '+A')}<br>+B: {_state_text(_r_and, '+B')}</p></div>"
    )


for _r_and in scored_and[:REPORT_TOP_N]:
    _r_and["detail_html"] = _detail_panel_html(_r_and)
print(f"details built for {min(REPORT_TOP_N, len(scored_and))} candidates")

## Report

In [ ]:
def _fmt_nt_pct_and(x, length):
    return "n/a" if x is None else f"{x:.2f}&nbsp;({100.0 * x / length:.0f}%)"


def _fmt_pp_and(x):
    return "n/a" if x is None else f"{x:.1f}"


def _fmt_ratio_and(r):
    if r["ratio"] is None:
        return "n/a"
    return f"{r['ratio']:.0f}" + ("*" if r["ratio_floored"] else "")


_CSS_and = r'''
@import url("https://cdn.jsdelivr.net/gh/aaaakshat/cm-web-fonts@latest/fonts.css");
@page { size: A4 landscape; margin: 16mm 14mm; }
body { font-family: "Computer Modern Serif", "CMU Serif", "Latin Modern Roman", "Times New Roman", serif;
       font-size: 11.5pt; line-height: 1.5; color: #111; max-width: 262mm; margin: 0 auto; }
h1 { font-size: 20pt; font-weight: normal; text-align: center; margin: 0 0 4pt 0; line-height: 1.25; }
.byline { text-align: center; font-size: 10pt; color: #444; margin-bottom: 18pt; }
h2 { font-size: 13.5pt; margin: 22pt 0 6pt 0; font-weight: bold; break-after: avoid; }
p { margin: 6pt 0; text-align: justify; }
dl { margin: 6pt 0; } dt { break-after: avoid; font-weight: bold; float: left; clear: left; width: 38mm; }
dd { break-inside: avoid; margin: 0 0 4pt 40mm; }
code { font-family: "Courier New", monospace; font-size: 9.5pt; }
table { border-collapse: collapse; margin: 8pt auto; font-size: 6.6pt; }
table.booktabs { border-top: 1.5pt solid #111; border-bottom: 1.5pt solid #111; }
table.booktabs th { border-bottom: 0.75pt solid #111; font-weight: bold; padding: 3pt 3pt; text-align: left; }
table.booktabs th.n { white-space: nowrap; }
table.booktabs td { padding: 2pt 1.5pt; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; }
figure { margin: 10pt 0; text-align: center; break-inside: avoid; }
figcaption { font-size: 10pt; color: #333; margin-top: 4pt; text-align: center; }
.note { font-size: 10pt; color: #333; }
.katex { font-size: 1.0em !important; } .katex-display { margin: 0.55em 0 !important; }
th.grp { text-align: center; border-bottom: 0.75pt solid #111; }
th.on { background: #fbe9e6; } th.off { background: #e6eef7; } th.ctl { background: #eef3e6; }
td.okk { background: #e3f3e6; } td.badk { background: #fbe3e0; }
.detail-link, .back, .detail-btn { white-space: nowrap; font-family: inherit; font-size: 9.5pt; color: #1a5fb4;
  text-decoration: underline; font-weight: normal; border: none; background: none; padding: 0; cursor: pointer; }
.back { margin-left: 14pt; }
.seqline { font-family: "Courier New", monospace; font-size: 9pt; word-break: break-all; line-height: 1.7; }
.dom { padding: 1pt 0; } .sw { display: inline-block; width: 8pt; height: 8pt; margin-right: 3pt; }
tr.detail > td { padding: 8pt 4pt 14pt 4pt; border-bottom: 0.75pt solid #999; }
.panel h3 { font-size: 12pt; margin: 0 0 6pt 0; } .state h4 { font-size: 11pt; margin: 6pt 0 2pt 0; }
.state { margin-top: 10pt; }
.bppRow { display: flex; flex-wrap: wrap; gap: 8pt; justify-content: center; margin-top: 10pt; }
.bppRow figure { flex: 0 0 45%; margin: 2pt; break-inside: avoid; } .bppRow img { max-width: 100%; }
td.seq { font-family: "Courier New", monospace; font-size: 8.5pt; word-break: break-all; }
table.small { font-size: 8.5pt; margin: 4pt 0; }
@media print { .detail, .detail-btn, .noprint { display: none !important; } }
.abstract { border-left: 3pt solid #1F6FEB; padding: 2pt 8pt; margin: 8pt 0 12pt 0; font-size: 11pt; background: #f4f7fb; }
.appendix .panel { break-before: page; } .appendix .panel:first-of-type { break-before: auto; } h2 + table { break-inside: avoid; } table.small, table.small tr { break-inside: avoid; } #results + p { break-after: avoid; break-inside: avoid; } .appendix .state { break-inside: avoid; }
.appendix svg { max-height: 112mm !important; }
.appendix .bppRow { margin-top: 4pt; }
'''

_TEMPLATE_and = r'''<!doctype html><html><head><meta charset="utf-8"><title>eu_and_v3_and report</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/contrib/auto-render.min.js"></script>
<script>document.addEventListener("DOMContentLoaded", function () {
  renderMathInElement(document.body, {delimiters: [
    {left: "\\[", right: "\\]", display: true}, {left: "\\(", right: "\\)", display: false}]});
});</script>
<script>function toggleDetail(id){var r=document.getElementById(id);r.hidden=!r.hidden;}</script>
<style>@@CSS@@</style></head><body>

<h1>AND screen</h1>
<div class="byline">@@SOURCE@@<br>@@POOL_BYLINE@@ fused in
all ordered pairs &middot; folded at @@TEMPERATURE@@&nbsp;&deg;C with ViennaRNA &middot; @@DATE@@</div>

<div class="abstract"><b>Summary.</b> @@ABSTRACT@@</div>

<h2>1&ensp;The construct and definitions</h2>
<p>An AND candidate fuses two single switches into one RNA. <b>Hairpin A</b> is the 5&prime; half: everything of the
first design before its own Kozak (prefix, toehold, stem, loop). <b>Hairpin B</b> is the 3&prime; half: everything of the
second design after its own prefix (toehold, stem, loop, Kozak and the payload that follows). A <b>spacer</b>
may sit between them (see below); both variants, <i>no_spacer</i> and <i>spacer</i>, are run. The pool is @@POOL_TEXT@@, taken in all
ordered pairs (A, B and B, A are both included, so each pair of hairpins appears twice, with the two orders of the hairpins) without self-pairs and without trigger pairs that are too similar (a run of more than
@@OVERLAP@@ nt of identity, or of complementarity with at most @@WOBBLE_PCT@@% G&middot;U wobble, between trigger A and trigger B).</p>
<p><b>The spacer.</b> A spacer is a short RNA inserted between hairpin A and hairpin B so that the two hairpins do not interfere with each other. It is not designed per pair from scratch: a fixed pool of @@SPACER_POOL@@ random candidate sequences is generated once (seed @@SPACER_SEED@@, so every run gives the same pool), with lengths cycling through @@SPACER_RANGE@@ nt and a bias towards A and U (35% each, 15% each G and C), because A&ndash;U pairs are weaker than G&ndash;C and a low-GC spacer is less likely to pair with anything. For each candidate pair, the candidate whose longest complementary run against the two toeholds and the two triggers is the shortest is chosen (ties: lower GC content, then pool order), i.e. the spacer that is least able to hybridise with them. The spacer is part of the fused RNA, so the numbers of the <i>spacer</i> variant include its effect; the <i>no_spacer</i> variant joins the halves directly.</p>
<dl>
<dt>single rank</dt><dd>@@SINGLE_RANK_TEXT@@ &ldquo;A, B&rdquo; in the table gives the single ranks of the
two designs fused, hairpin A first. Their sequences are listed in each candidate's details.</dd>
<dt>prefix, Kozak, payload</dt><dd>a switch RNA starts with a short <i>prefix</i>, then the toehold and the hairpin; the <i>Kozak</i> sequence follows the hairpin and the
<i>payload</i> (the coding sequence) follows the Kozak. In the fused construct only the 5&prime; half's prefix remains (the 3&prime; half starts after its own prefix) and only
the 3&prime; half's payload; the payload is folded but not drawn.</dd>
<dt>how to read the figures</dt><dd>2D structures: every circle is a nucleotide coloured by part (hairpin A and B use different hues); a <b>gold ring</b> marks a nucleotide paired into the payload
(the undrawn &ldquo;hidden tail&rdquo;) and a <b>pink ring</b> one paired to an undrawn trigger strand. Heat maps: the axes run over the whole folded RNA, including the payload and the triggers
present in that state; the dotted red lines mark where one strand ends and the next begins.</dd>
<dt>trigger A, B</dt><dd>the real triggers of the two designs; trigger A targets hairpin A, trigger B hairpin B.</dd>
<dt>stem</dt><dd>each hairpin has a stem with two sides of \(L_A\) (resp. \(L_B\)) nucleotides, \(\mathrm{stem}_{\uparrow}\) (5&prime; side) and
\(\mathrm{stem}_{\downarrow}\) (3&prime; side), paired with each other in the closed hairpin. Here \(L_A\), \(L_B\) take the values @@STEM_LENS@@.
\(L=\min(L_A,L_B)\) is used for percentages (section 2).</dd>
<dt>toehold, hairpin, loop</dt><dd>the <i>toehold</i> is the single-stranded stretch where a trigger first binds; a <i>hairpin</i> is a stem (\(\mathrm{stem}_{\uparrow}\) paired with \(\mathrm{stem}_{\downarrow}\)) closed by a <i>loop</i> of unpaired nucleotides. Binding the trigger is meant to open the hairpin.</dd>
<dt>pp</dt><dd>percentage points, the unit of a difference of two percentages.</dd>
<dt>state</dt><dd>the strands folded together: <i>alone</i> (the fused switch), <i>+A</i>, <i>+B</i>, <i>+A+B</i> (with the named trigger(s)), and @@N_CONTROLS@@ <i>control</i> states (each with two mismatched triggers).</dd>
<dt>mismatched trigger</dt><dd>the real trigger of another design of the first @@N_SINGLE@@ CSV rows (or, if none qualifies, another sequence offered for this purpose) that shares at most @@CONTROL_OVERLAP@@ nt of identical sequence with either real trigger; @@N_CONTROLS@@ pairs of them per candidate are used as wrong inputs in the controls.</dd>
<dt>structure</dt><dd>one folded arrangement \(s\) of the strands, i.e. a set of base pairs.</dd>
<dt>\(E(s)\), \(G\)</dt><dd>the energy of \(s\), and the free energy of the whole ensemble of structures of the state (kcal/mol).</dd>
<dt>\(P(s)\)</dt><dd>the probability of \(s\), \(P(s)=\exp\!\big(-(E(s)-G)/RT\big)\), with \(R\) the gas constant and \(T\) the absolute temperature.</dd>
<dt>open nucleotide</dt><dd>a \(\mathrm{stem}_{\uparrow}\) nucleotide of hairpin \(h\) not paired to a \(\mathrm{stem}_{\downarrow}\) nucleotide of the same hairpin in \(s\).</dd>
<dt>\(k_h(s)\)</dt><dd>the number of open nucleotides of hairpin \(h\in\{A,B\}\) in \(s\), anywhere on its stem.</dd>
<dt>\(r_h(s)\)</dt><dd>the longest stretch of <i>consecutive</i> open nucleotides of hairpin \(h\) in \(s\). Always \(r_h\le k_h\).</dd>
<dt>\(\Delta G\)</dt><dd>\(G_{+A+B}-\min\big(G_{\text{alone}}+G_A+G_B,\;G_{+A}+G_B,\;G_{+B}+G_A\big)\), kcal/mol, from the exact partition function, where \(G_A,G_B\) are the free energies of the triggers folded alone: the free energy gained by forming the two-trigger complex relative to the most stable OFF state with its missing triggers kept separate. More negative: the pair of triggers is favoured more strongly. Orders only the tied group (note that a more stable hairpin makes \(\Delta G\) less negative, so ordering by it slightly favours less stable switches). Strand concentrations are not included (the duplex-initiation term of ViennaRNA's dimer energy is).</dd>
</dl>

<h2>2&ensp;Open, ON and OFF</h2>
<p><b>ON (+A+B).</b> In a structure \(s\), hairpin A has a longest run \(r_A(s)\) of consecutive open nucleotides and hairpin B a longest run \(r_B(s)\). For each structure the two runs are added, divided by the total stem length \(L_A+L_B\) and weighted with the probability of the structure; summed over all structures this is the expectation of the sum of open nucleotides of both stems,
\[ \mathrm{Open}_{+A+B}=\mathbb{E}[r_A+r_B]=\sum_{s}P(s)\,\big(r_A(s)+r_B(s)\big),\qquad
\mathrm{ON}=100\,\frac{\mathrm{Open}_{+A+B}}{L_A+L_B}=100\,\sum_{s}P(s)\,\frac{r_A(s)+r_B(s)}{L_A+L_B}. \]
Since an expectation of a sum is the sum of the expectations, \(\mathrm{Open}_{+A+B}=\mathbb{E}[r_A]+\mathbb{E}[r_B]\). ON is 100% when both stems are open over their full length and about 50% when only one is. The two hairpins need not be open in the same structure.</p>
<p><b>OFF.</b> For the states that should stay closed the quantity is the expected longest run that is open in <i>both</i> hairpins at once, \(\mathrm{Open}=\mathbb{E}[\min(r_A,r_B)]=\sum_{s}P(s)\min\big(r_A(s),r_B(s)\big)\), as a percentage of the shorter stem \(L=\min(L_A,L_B)\), and OFF is the worst (largest) over the states:
\[ \mathrm{OFF}=100\,\frac{\max\big(\mathrm{Open}_{\text{alone}},\mathrm{Open}_{+A},\mathrm{Open}_{+B},\mathrm{Control}\big)}{L}. \]
(In the +A state hairpin A is meant to open but hairpin B should not, so the minimum is small; likewise for +B. The Control is defined below.) The two percentages have different denominators, the two stems together for ON and the shorter stem for OFF, and ON is a sum while OFF is a minimum: ON rewards each hairpin that responds to its own trigger, OFF flags any state in which both hairpins are open at once. The report lists the min-based \(\mathrm{ON}_{\min}=100\,\mathbb{E}[\min(r_A,r_B)]_{+A+B}/L\) and \(\mathrm{ON}_{\min}-\mathrm{OFF}\) beside it, which compares like with like.</p>
<p>The candidates are compared with \(\mathrm{ON}-\mathrm{OFF}\) (percentage points) and \(\mathrm{ratio}=\mathrm{ON}/\max(\mathrm{OFF},@@FLOOR@@)\). <b>They are ranked by \(\mathrm{ON}-\mathrm{OFF}\)</b> (sum-based ON), largest first, with near-ties handled as described below. A percentage of the stem(s) makes candidates with different stem lengths comparable (for OFF the effect may go the other way, since fraying at the stem ends is roughly constant in nucleotides; stem lengths are not fully neutral). The ratio is listed for information and orders nothing; the floor of @@FLOOR@@% on OFF keeps it finite, and a ratio computed with the floor is marked with an asterisk.</p>
<p><b>Controls.</b> To see whether Open responds only to the <i>right</i> triggers, every candidate is also folded in @@N_CONTROLS@@ control states, each with two mismatched triggers in place of A and B (real triggers of other designs sharing at most @@CONTROL_OVERLAP@@ nt of identity with either real trigger). Because the sampler is unreliable for mismatched states, each control is taken from the exact matrix as the upper bound \(\min(\mathbb{E}[k_A],\mathbb{E}[k_B])\ge\mathrm{Open}\), and the largest of the @@N_CONTROLS@@ is the <i>Control</i>. OFF is the largest of Open(alone), Open(+A), Open(+B) and the Control (all in %), so a construct that mismatched triggers can open is penalised.</p>
<p><b>Ties.</b> Repeating the sampling with another seed changes ON &minus; OFF by a small amount (section 5). Candidates within the 95th percentile of that change from the best one cannot be told apart from it; they form one <i>tied group</i> (&dagger; in the table), ranked inside it by \(\Delta G\) (more negative first) and listed first. Only Monte-Carlo noise is counted; model error is larger, so candidates just outside the group are not clearly worse. The tied group is the shortlist; inside it the order is by \(\Delta G\), so rank 1 need not have the highest ON &minus; OFF. The two orders of one pair (A, B and B, A) and the two spacer variants of a pair are separate rows and often tie.</p>
<p><b>What Open does and does not measure.</b> The +A+B state is the ensemble with both triggers present as a complex; trigger concentrations, binding kinetics and the fraction of constructs that form the complex are not modelled. Open measures how far the hairpin stems are opened at equilibrium (the exact ensemble of ViennaRNA at the stated temperature). It is a structural proxy
for the hairpins releasing the regions behind them; it is not a measurement of translation, of ribosome binding or start-codon accessibility, of kinetics, or of behaviour in a cell, and trigger accessibility and self-structure, cross-talk between triggers, the openness of the stem_down sides and of the start codon, temperature or parameter-set sensitivity, and the origin of the CSV are not assessed; hairpin A is treated as a second input whose opening is required, its mechanism of gating the payload is assumed, not modelled; the only control is one pair of mismatched triggers per candidate (no scrambled triggers or known reference construct). The pool is preselected (the top of the CSV, then the best singles by the same kind of metric), so the candidates recombine a few hairpins.</p>
<p>Next to each Open the report lists the expected number of open nucleotides anywhere on the stems: \(\mathbb{E}[k_A+k_B]\) for +A+B and \(\mathbb{E}[\min(k_A,k_B)]\) for the
other states (\(\mathrm{Open}\le\mathbb{E}[k]\) in every case). It is the more relevant number for the states that should stay closed: stems open in scattered places have a small
Open but may have a larger \(\mathbb{E}[k]\).</p>

<h2>3&ensp;Estimation and verification</h2>
<p><i>In plain words:</i> Open is estimated by drawing random structures with the probability they have at equilibrium. The exact base-pair probability matrix cannot give Open but can give the average number of open nucleotides, so it is used to check that the drawn structures behave correctly.</p>
<p><b>Sampled estimate.</b> The sum over all structures cannot be written out, so \(N\) structures \(S_1,\dots,S_N\) are drawn from the ensemble
(ViennaRNA stochastic backtracking, <code>FoldEngine.sample_structures</code>) and averaged, for example
\[ \mathrm{Open}_{+A+B}\approx\frac1N\sum_{i=1}^{N}\big(r_A(S_i)+r_B(S_i)\big),\qquad N=@@N_SAMPLES@@ . \]
A structure is drawn with probability \(P(s)\), so this estimates the full sum, including structures that are never drawn. <b>No
normalisation is needed:</b> the draws are already distributed according to \(P(s)\). Its statistical error is \(\mathrm{sd}/\sqrt N\).</p>
<p><b>Matrix check.</b> The base-pair probability matrix \(p_{ij}\) gives each hairpin's expected number of open nucleotides exactly,
\[ \mathbb{E}[k_h]=\sum_{i\in \mathrm{stem}^h_{\uparrow}}\Big(1-\sum_{j\in \mathrm{stem}^h_{\downarrow}}p_{ij}\Big),\qquad h\in\{A,B\}. \]
It cannot give the longest runs, because it does not know which positions are open together in the same structure. It does bound them, since \(r_h\le k_h\):
\(\mathrm{Open}_{+A+B}\le\mathbb{E}[k_A]+\mathbb{E}[k_B]\) and, for the other states, \(\mathrm{Open}\le\min(\mathbb{E}[k_A],\mathbb{E}[k_B])\). A state is therefore
<i>verified</i> (its \(\mathbb{E}[k]\) cells are shaded light green in Appendix A) when (1) the sampled \(\mathbb{E}[k_A]\) and \(\mathbb{E}[k_B]\) each agree with the
matrix within \(\tau_A\), \(\tau_B\), which shows the sampler reproduces the per-nucleotide openness of that state to within \(\tau\) (it does not prove the longest runs themselves), and (2) its Open does not exceed the matrix bound by more than the corresponding \(\tau\); otherwise it is shaded light red.</p>
<p><b>The tolerance.</b> For each hairpin, \(\tau_h=@@VERIFY_SIGMA@@\,\mathrm{sd}(k_h)/\sqrt N+@@VERIFY_FLOOR@@\) nucleotides, where \(\mathrm{sd}(k_h)\) is the standard deviation of \(k_h\) over the \(N\) draws of that state: a state that should stay closed is checked tightly, a variable one loosely. The constant only allows for the rounding of the matrix. Verification catches gross sampler faults; \(\tau\) can be as large as the seed-to-seed noise of the ranking, so green does not guarantee ranking-level accuracy. The tolerance sets the shading only, not any Open or \(\mathbb{E}[k]\) value.</p>
<p><b>Single-input leakage.</b> Because OFF uses the minimum of the two runs, a hairpin that opens on its own while the other stays closed does not raise OFF (it counts only if the other hairpin is open too). This relies on hairpin A actually gating expression, which is assumed here. The per-hairpin E[k] values in the details show such leakage; the junction between the two halves can also change a hairpin's behaviour compared with its single switch.</p>
<p><b>Check for +A+B.</b> For the +A+B state the matrix bounds the sum of the two longest runs, \(\mathbb{E}[r_A+r_B]\le\mathbb{E}[k_A]+\mathbb{E}[k_B]\), which is exactly the quantity ON uses; the per-hairpin agreement of the sampled and exact \(\mathbb{E}[k]\) is the other part of the check.</p>
<p><b>Unverified states.</b> States that fail the check stay in all medians, noise figures and rank correlations; they are listed in section 5. The sampler's dependence on strand order is empirical, not explained.</p>
<p><b>Limitation.</b> ViennaRNA's sampler can return wrong draws that depend on the order in which the strands are written. Here the trigger
of the hairpin nearest the 5&prime; end (A) is written before the fused strand and the trigger of the 3&prime; hairpin (B) after it; this order was
found to match the matrix, but nothing is filtered or corrected. In this run @@N_OK@@ states passed the check (alone / +A / +B / +A+B); a state shaded light red should not be trusted (biased upwards: the wrong draws always have the stems open). @@RED_TEXT@@</p>

<h2>4&ensp;Parameters</h2>
<table class="booktabs"><tr><th>name</th><th>value</th><th>meaning</th></tr>
@@PARAMS@@
</table>

<h2>5&ensp;Summary</h2>
<table class="booktabs">
<tr><td>candidates scored</td><td class="n">@@N_SCORED@@</td></tr>
<tr><td>excluded: self-pairs / trigger pairs too similar / fuse failed (the two halves could not be joined into one RNA)</td><td class="n">@@N_EXCL@@</td></tr>
<tr><td>states verified against the matrix: alone / +A / +B / +A+B</td><td class="n">@@N_OK@@</td></tr>
<tr><td>median ON (% of both stems) / OFF (% of the shorter stem)</td><td class="n">@@MED_ON@@% / @@MED_OFF@@%</td></tr>
<tr><td>median ON &minus; OFF (percentage points)</td><td class="n">@@MED_DIFF@@</td></tr>
<tr><td>median control (worst of @@N_CONTROLS@@ mismatched states, matrix upper bound), % of the shorter stem</td><td class="n">@@MED_CTRL@@%</td></tr>
<tr><td>seed-to-seed change of ON &minus; OFF, second independent run: median / 95th percentile (pp)</td><td class="n">@@NOISE_MED@@ / @@NOISE_P95@@</td></tr>
<tr><td>ranking by the sum (used) vs the min definition of ON: rank correlation / overlap of the top @@REPORT_TOP_N@@</td><td class="n">@@RANK_AGREE@@ / @@TOP_OVERLAP@@</td></tr>
<tr><td>states not verified (candidate rank, state)</td><td class="n">@@UNVERIFIED@@</td></tr>
</table>
<p>@@BEST_TEXT@@</p>

<h2 id="results">6&ensp;First @@REPORT_TOP_N@@ candidates (tied group first, then by \(\mathrm{ON}-\mathrm{OFF}\))</h2>
<p class="note">Green or red shading only says whether the sampler agrees with the matrix; it is not a quality flag for the design. &dagger; = member of the tied group (section 2). * after a ratio = OFF was below the floor. A and B are the single-report ranks of the two designs fused (see the details). \(L\) is the stem length of hairpin A / B.
For +A+B, <i>Open sum (ON)</i> is \(\mathbb{E}[r_A+r_B]\), the expected sum of the longest open runs of the two stems, in nucleotides with the percentage of \(L_A+L_B\) in brackets; The expected number of open nucleotides of both stems, \(\mathbb{E}[k_A+k_B]\), sampled and exact (matrix), is in Appendix A at the end. <i>Open min</i> is \(\mathbb{E}[\min(r_A,r_B)]\) (percentage of the shorter stem), listed for information only.
The ranking is by \(\mathrm{ON}-\mathrm{OFF}\) (sum-based ON, OFF as defined in section 2). The Control is the largest exact upper bound over @@N_CONTROLS@@ constructs with two mismatched triggers instead of A and B; it is part of OFF. The sampled \(\mathbb{E}[\min(k_A,k_B)]\) of the OFF states and the sampler verification are in <a href="#appendixA">Appendix A</a>; the per-hairpin matrix values are in the details. OFF is the largest of Open(alone), Open(+A), Open(+B) and the Control.</p>
<table class="booktabs">
<thead><tr><th rowspan="2" class="n">rank</th><th rowspan="2" style="white-space:nowrap">single ranks<br>A,&nbsp;B</th><th rowspan="2">spacer</th><th rowspan="2" class="n">\(L_A/L_B\)</th>
<th colspan="2" class="grp on">ON &ensp;<small>+A+B</small></th>
<th colspan="3" class="grp off">OFF &ensp;<small>alone, +A, +B</small></th>
<th class="grp ctl">Control &ensp;<small>5 mismatched states</small></th>
<th rowspan="2" class="n">ON &minus; OFF<br>(pp)</th><th rowspan="2" class="n">ON<sub>min</sub> &minus; OFF<br>(pp, info)</th><th rowspan="2" class="n">ratio<br>ON / OFF</th><th rowspan="2" class="n">\(\Delta G\)<br>(kcal/mol)<br>lower = tighter</th>@@BTNHEAD@@</tr>
<tr><th class="n">\(\mathrm{Open}_{+A+B}\)<br>sum (ON)</th><th class="n">\(\mathrm{Open}_{+A+B}\)<br>min (info)</th>
<th class="n">\(\mathrm{Open}_{\text{alone}}\)</th><th class="n">\(\mathrm{Open}_{+A}\)</th><th class="n">\(\mathrm{Open}_{+B}\)</th>
<th class="n">\(\min\mathbb{E}[k]\)<br>matrix (&ge; Open)</th></tr></thead>
@@ROWS@@
</table>
@@APPENDIX@@
@@APPENDIX2@@
</body></html>'''


def _row_html_and(r, interactive=True):
    L = r["l_min"]
    res = r["res"]
    cls = {s: ("okk" if r["ok"][s] else "badk") for s in _STATES}
    on_k_sampled = res["+A+B"]["k_a_sampled"] + res["+A+B"]["k_b_sampled"]
    on_k_matrix = res["+A+B"]["k_a_matrix"] + res["+A+B"]["k_b_matrix"]
    rid = f"d{r['our_rank']}"
    button = (f"<td class='noprint'><button class='detail-btn' onclick=\"toggleDetail('{rid}')\">"
              "Details</button></td>") if interactive else (
        f"<td><a class='detail-link' href='#p{r['our_rank']}'>Details</a></td>")
    detail_row = (f"<tr class='detail' id='{rid}' hidden><td colspan='15'>{r['detail_html']}</td></tr>"
                  if interactive and "detail_html" in r else "")
    spacer = f"spacer ({len(r['spacer_seq'])}&nbsp;nt)" if r["spacer_seq"] else "no_spacer"
    cells = (
        f"<td class='n'>{r['our_rank']}{'&dagger;' if r['tied'] else ''}</td>"
        f"<td style='white-space:nowrap'>{r['d5']['single_rank']},&nbsp;{r['d3']['single_rank']}</td><td>{spacer}</td>"
        f"<td class='n'>{r['l_a']}/{r['l_b']}</td>"
        f"<td class='n'>{_fmt_nt_pct_and(r['open']['+A+B'], r['l_a'] + r['l_b'])}</td>"
        f"<td class='n'>{_fmt_nt_pct_and(res['+A+B']['open_sampled'], L)}</td>"
    )
    for s in ("alone", "+A", "+B"):
        cells += f"<td class='n'>{_fmt_nt_pct_and(r['open'][s], L)}</td>"
    cells += (f"<td class='n'>{_fmt_nt_pct_and(r['ctrl_pct'] * L / 100.0, L) if r['ctrl_pct'] is not None else 'n/a'}</td>")
    cells += (f"<td class='n'><b>{_fmt_pp_and(r['diff_pp'])}</b></td><td class='n'>{_fmt_pp_and(r['diff_min_pp'])}</td>"

              f"<td class='n'>{_fmt_ratio_and(r)}</td>"
              f"<td class='n'>{r['dg']:.1f}</td>")
    return f"<tr>{cells}{button}</tr>{detail_row}"


_stem_text_and = ", ".join(str(n) for n in sorted({x for r in scored_and for x in (r["l_a"], r["l_b"])}))
_param_rows_and = [
    ("SPACER_CANDIDATE_POOL_SIZE", SPACER_CANDIDATE_POOL_SIZE, "random candidate spacers generated once (A/U-biased, seeded)"),
    ("SPACER_LENGTHS", f"{SPACER_MIN_LENGTH}&ndash;{SPACER_MAX_LENGTH}", "spacer lengths (nt), cycled through the pool"),
    ("SPACER_SEED", SPACER_SEED, "seed of the spacer pool"),
    ("N_SINGLE", N_SINGLE, "first CSV rows scored as single switches"),
    ("AND_TOP_N", AND_TOP_N, "top designs of the single report (or, without it, top singles by ON &minus; OFF) fused in all ordered pairs"),
    ("SPACER_VARIANTS", ", ".join(SPACER_VARIANTS), "both variants are run and ranked together"),
    ("FAKE_MAX_SEQ_OVERLAP_NT", FAKE_MAX_SEQ_OVERLAP_NT,
     "trigger A and trigger B may share at most this many nucleotides in a row (identity or complementarity)"),
    ("WOBBLE_MAX_FRACTION", WOBBLE_MAX_FRACTION, "max share of G&middot;U wobble in a counted complementary run"),
    ("TEMPERATURE_C", TEMPERATURE_C, "folding temperature, &deg;C"),
    ("N_SAMPLES", N_SAMPLES, "structures drawn from the ensemble for each state"),
    ("SEED", SEED, "random seed for the sampler (a rerun repeats)"),
    ("CHECK_SEED_OFFSET", CHECK_SEED_OFFSET, "a second sampling run uses SEED + this; it only measures the noise (section 5)"),
    ("VERIFY_SIGMA", VERIFY_SIGMA, "&tau;: allowed disagreement with the matrix, in standard errors of the sampled mean"),
    ("VERIFY_FLOOR_NT", VERIFY_FLOOR_NT, "added to &tau; (nt)"),
    ("N_CONTROLS", N_CONTROLS, "mismatched control states per candidate (two mismatched triggers each); OFF is the worst of alone, +A, +B and these"),
    ("CONTROL_MAX_OVERLAP_NT", CONTROL_MAX_OVERLAP_NT, "each mismatched control trigger shares at most this many nt of identity with either real trigger"),
    ("OFF_FLOOR_PCT", OFF_FLOOR_PCT, "OFF is not allowed below this many % of the stem in the ratio ON / OFF"),
    ("REPORT_TOP_N", REPORT_TOP_N, "candidates listed in the table of section 6"),
]
_params_html_and = "".join(
    f"<tr><td><code>{n}</code></td><td>{v}</td><td>{m}</td></tr>" for n, v, m in _param_rows_and
)
_valid = [r for r in scored_and if r["diff_pp"] is not None]
_n_and = len(scored_and)
_fields_and = {
    "CSS": _CSS_and, "SOURCE": html.escape(Path(CSV_PATH).name),
    "ABSTRACT": (f"{_n_and} AND constructs (two single switches fused, A at the 5&prime; end, B at the 3&prime; end) are folded alone, with trigger A, with trigger B, with both and with mismatched triggers (controls). "
        f"<b>ON</b> is the expected sum of the longest open runs of the two stems with both triggers, as a percentage of the two stem lengths together; <b>OFF</b> is the worst case of the expected longest run open in both hairpins at once, over alone, +A, +B and 5 mismatched-trigger controls, as a percentage of the shorter stem (100% = fully open). A good AND has ON near 100% and OFF near 0%. "
        f"Median ON is {statistics.median(r['on_pct'] for r in _valid):.1f}% and median OFF {statistics.median(r['off_pct'] for r in _valid):.1f}%. "
        f"{n_tied_with_best_and} of {_n_and} candidates lie within the run-to-run difference (95th percentile of the change of ON &minus; OFF between two independent samplings: {noise_p95_and:.2f} points) of the best. "
        f"They are marked &dagger; and are the shortlist (ordered by \\(\\Delta G\\)); any of them is as good as the others by this score. Ranking by the min-based ON instead gives a similar order (Spearman rank correlation {rank_agreement:.2f}; the overlap of the top {REPORT_TOP_N} in section 5 is the more informative figure). Structure-based prediction only, no experimental calibration."), "N_SINGLE": N_SINGLE, "AND_TOP_N": AND_TOP_N,
    "POOL_BYLINE": (f"Top {AND_TOP_N} designs of the single report ({html.escape(Path(_single_csv).name)})" if _single_csv is not None
                    else f"First {N_SINGLE} designs of the CSV, top {AND_TOP_N} by single-switch ON &minus; OFF"),
    "POOL_TEXT": (f"the {AND_TOP_N} best designs of the single report <code>{html.escape(Path(_single_csv).name)}</code> (its tied group first, then by ON &minus; OFF; the same ranking as that report)" if _single_csv is not None
                  else f"the {AND_TOP_N} best single switches by ON &minus; OFF, scored here"),
    "SINGLE_RANK_TEXT": ("the rank of a design in the single report (the pool is its top designs);" if _single_csv is not None
                         else "the position of a design when the first rows are ordered by ON &minus; OFF alone (without the tie-group ordering of the single report, so it can differ from that report's rank);"),
    "TEMPERATURE": f"{TEMPERATURE_C:g}", "DATE": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "SPACER_RANGE": f"{SPACER_MIN_LENGTH}&ndash;{SPACER_MAX_LENGTH}", "OVERLAP": FAKE_MAX_SEQ_OVERLAP_NT, "SPACER_POOL": SPACER_CANDIDATE_POOL_SIZE, "SPACER_SEED": SPACER_SEED,
    "WOBBLE_PCT": f"{100 * WOBBLE_MAX_FRACTION:.0f}", "STEM_LENS": _stem_text_and, "FLOOR": f"{OFF_FLOOR_PCT:g}", "CONTROL_OVERLAP": CONTROL_MAX_OVERLAP_NT,
    "N_SAMPLES": N_SAMPLES, "VERIFY_SIGMA": f"{VERIFY_SIGMA:g}", "VERIFY_FLOOR": f"{VERIFY_FLOOR_NT:g}",
    "RED_TEXT": ("All listed candidates are shaded green." if not any(not r["ok"][s] for r in scored_and[:REPORT_TOP_N] for s in _STATES)
                 else "Some listed candidates have a red cell."),
    "PARAMS": _params_html_and, "N_SCORED": _n_and,
    "N_EXCL": f"{_n_self} / {_n_similar} / {_n_fuse_failed}",
    "N_OK": " / ".join(f"{n_ok[s]}" for s in _STATES) + f" of {_n_and}",
    "N_CONTROLS": N_CONTROLS,
    "MED_CTRL": f"{statistics.median(r['ctrl_pct'] for r in _valid if r['ctrl_pct'] is not None):.1f}",
    "MED_ON": f"{statistics.median(r['on_pct'] for r in _valid):.1f}",
    "MED_OFF": f"{statistics.median(r['off_pct'] for r in _valid):.1f}",
    "MED_DIFF": f"{statistics.median(r['diff_pp'] for r in _valid):.1f}",
    "REPORT_TOP_N": REPORT_TOP_N,
    "NOISE_MED": f"{noise_median_and:.2f}", "NOISE_P95": f"{noise_p95_and:.2f}",
    "RANK_AGREE": f"{rank_agreement:.3f}", "TOP_OVERLAP": f"{top_overlap} of {REPORT_TOP_N}",
    "UNVERIFIED": ", ".join(f"rank {rk}: {st}" for rk, st in _unverified) or "none",
    "BEST_TEXT": (f"Highest ON &minus; OFF: {max(r['diff_pp'] for r in scored_and if r['diff_pp'] is not None):.1f} points. "
                  f"{n_tied_with_best_and} of the {len(scored_and)} candidates lie within the 95th-percentile seed-to-seed change "
                  f"({noise_p95_and:.2f} points) of it. They form the tied group (&dagger;), the shortlist (these are {len({(r['d5']['single_rank'], r['d3']['single_rank']) for r in scored_and if r['tied']})} ordered pairs of hairpins, {len({frozenset((r['d5']['single_rank'], r['d3']['single_rank'])) for r in scored_and if r['tied']})} unordered, each with up to two spacer variants, so they are far fewer than {n_tied_with_best_and} independent designs); the table lists only the first {REPORT_TOP_N} of them, the others carry tied_group = True in the CSV; inside it candidates are ordered by \\(\\Delta G\\), not by ON &minus; OFF. "
                  f"A state that is not verified (see the list above) has an unreliable ON or OFF; its row is shaded red in the table."),
}


def _fill_and(template, fields):
    for _key, _value in fields.items():
        template = template.replace(f"@@{_key}@@", str(_value))
    return template


_top_and = scored_and[:REPORT_TOP_N]


def _appx_row(r):
    res = r["res"]["+A+B"]
    cls = "okk" if r["ok"]["+A+B"] else "badk"
    flags = " ".join(("&#10003;" if r["ok"][st_] else "&times;") for st_ in _STATES)
    return (f"<tr><td class='n'>{r['our_rank']}</td>"
            f"<td style='white-space:nowrap'>{r['d5']['single_rank']},&nbsp;{r['d3']['single_rank']}</td>"
            f"<td class='n'>{r['l_a']}/{r['l_b']}</td>"
            f"<td class='n {cls}'>{_fmt_nt_and(res['k_a_sampled'] + res['k_b_sampled'])}</td>"
            f"<td class='n {cls}'>{_fmt_nt_and(res['k_a_matrix'] + res['k_b_matrix'])}</td>"
            f"<td class='n'>{_fmt_nt_and(r['open']['+A+B'])}</td>"
            + "".join(f"<td class='n {'okk' if r['ok'][st_] else 'badk'}'>{_fmt_nt_and(r['res'][st_]['k_min_sampled'])}</td>"
                      for st_ in ("alone", "+A", "+B"))
            + f"<td>{flags}</td></tr>")


_appendix2 = ("<div class='appendix2'><h2 id='appendixA'>Appendix A&ensp;Expected number of open nucleotides and sampler check</h2>"
              "<p class='note'>For the +A+B state: \\(\\mathbb{E}[k_A+k_B]\\) is the expected number of open nucleotides "
              "anywhere on the two stems (not necessarily consecutive), sampled and from the exact matrix; Open sum is the quantity used for ON, "
              "\\(\\mathbb{E}[r_A+r_B]\\) (it cannot exceed \\(\\mathbb{E}[k_A+k_B]\\)). For the OFF states (alone, +A, +B) the sampled \\(\\mathbb{E}[\\min(k_A,k_B)]\\), which bounds their Open, is listed. A cell is shaded light green when the state's sampled \\(\\mathbb{E}[k]\\) agrees with the matrix within the tolerance (section 3), light red otherwise. The last column shows the verification of the four states "
              "(alone, +A, +B, +A+B in this order; &#10003; = verified).</p>"
              "<table class='booktabs'><thead><tr><th class='n'>rank</th><th>single ranks<br>A,&nbsp;B</th><th class='n'>\\(L_A/L_B\\)</th>"
              "<th class='n'>\\(\\mathbb{E}[k_A+k_B]\\)<br>sampled</th><th class='n'>\\(\\mathbb{E}[k_A+k_B]\\)<br>matrix</th>"
              "<th class='n'>Open sum<br>\\(\\mathbb{E}[r_A+r_B]\\)</th>"
              "<th class='n'>alone<br>\\(\\mathbb{E}[\\min(k_A,k_B)]\\)</th><th class='n'>+A<br>\\(\\mathbb{E}[\\min(k_A,k_B)]\\)</th><th class='n'>+B<br>\\(\\mathbb{E}[\\min(k_A,k_B)]\\)</th>"
              "<th>states verified</th></tr></thead>"
              + "".join(_appx_row(r) for r in _top_and) + "</table></div>")
report_html_and = _fill_and(_TEMPLATE_and, {
    **_fields_and, "BTNHEAD": '<th rowspan="2" class="noprint"></th>', "APPENDIX": "", "APPENDIX2": _appendix2,
    "ROWS": "".join(_row_html_and(r) for r in _top_and),
})
_appendix_and = ("<div class='appendix'><h2>7&ensp;Candidate details</h2><p class='note'>Five pages per candidate "
             "of the table above: its parts, the 2D structure of the OFF state (alone), the 2D structure of the "
             "ON state (+A+B), and the base-pair probabilities of all four states. (The HTML version shows the same "
             "under each row's Details button.)</p>" + "".join(r["detail_html"] for r in _top_and) + "</div>")
report_print_html_and = _fill_and(_TEMPLATE_and, {
    **_fields_and, "BTNHEAD": '<th rowspan="2"></th>', "APPENDIX": _appendix_and, "APPENDIX2": _appendix2,
    "ROWS": "".join(_row_html_and(r, interactive=False) for r in _top_and),
})

_ts_and = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
report_html_path_and = _report_root / f"eu_and_v3_and_report_{_ts_and}.html"
report_html_path_and.write_text(report_html_and)
_print_html_path_and = _report_root / f"eu_and_v3_and_report_{_ts_and}_print.html"
_print_html_path_and.write_text(report_print_html_and)

import csv as _csv
all_csv_path_and = _report_root / f"eu_and_v3_and_all_candidates_{_ts_and}.csv"
with all_csv_path_and.open("w", newline="") as _fh_and:
    _w = _csv.writer(_fh_and)
    _w.writerow(["our_rank", "csv_rank_A", "csv_rank_B", "spacer", "L_A", "L_B", "Open_min_alone_nt",
                 "Open_min_+A_nt", "Open_min_+B_nt", "Open_sum_+A+B_nt", "E_k_sum_+A+B_nt", "ON_pct", "OFF_pct",
                 "OFF_source", "ON_minus_OFF_pp", "ONmin_minus_OFF_pp", "ratio_ON_over_OFF", "dG_kcal_per_mol",
                 "verified_alone", "verified_+A", "verified_+B", "verified_+A+B", "tied_group",
                 "control_matrix_upper_bound_pct", "trigger_len_A", "trigger_len_B", "spacer_len"])
    for r in scored_and:
        _w.writerow([r["our_rank"], r["d5"]["rank"], r["d3"]["rank"], r["spacer_label"], r["l_a"],
                     r["l_b"], r["open"]["alone"], r["open"]["+A"], r["open"]["+B"], r["open"]["+A+B"],
                     r["res"]["+A+B"]["k_a_sampled"] + r["res"]["+A+B"]["k_b_sampled"], r["on_pct"], r["off_pct"], r["off_source"],
                     r["diff_pp"], r["diff_min_pp"], r["ratio"], r["dg"], *(r["ok"][s] for s in _STATES), r["tied"],
                     r["ctrl_pct"],
                     len(r["d5"]["trigger"]), len(r["d3"]["trigger"]), len(r["spacer_seq"])])
print(f"wrote {all_csv_path_and}")


def _render_pdf_and(html_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        script = base / "src/engine/gates/notebooks/toehold/render_pdf.py"
        if script.exists():
            break
    else:
        print("render_pdf.py not found -- HTML written, PDF skipped")
        return None
    pdf_path = html_path.with_suffix(".pdf")
    subprocess.run([sys.executable, str(script), str(html_path), str(pdf_path)], check=True)  # landscape
    print(f"wrote {pdf_path}")
    return pdf_path


_pdf_made_and = _render_pdf_and(_print_html_path_and)   # PDF: table + the details of every candidate
report_pdf_path_and = _pdf_made_and.rename(report_html_path_and.with_suffix(".pdf")) if _pdf_made_and else None
_print_html_path_and.unlink()   # only the PDF's source
print(f"HTML report with Details buttons -> {report_html_path_and}")

## Data for an interactive page

Writes the rows of all candidates (`rows.json`) and the detail panels of the best `ARTIFACT_DETAIL_N` candidates (small scripts in `d/`) next to the report, for a page with filters and expandable structures and heat maps.

In [ ]:
if EXPORT_ARTIFACT:
    import gzip as _gzip
    _art_dir = _report_root / f"eu_and_v3_and_artifact_{_ts_and}"
    (_art_dir / "d").mkdir(parents=True, exist_ok=True)
    _rows = []
    for _r_and in scored_and:
        _res = _r_and["res"]
        _rows.append({
            "rank": _r_and["our_rank"], "tied": bool(_r_and["tied"]),
            "A": _r_and["d5"]["single_rank"], "B": _r_and["d3"]["single_rank"],
            "csvA": int(_r_and["d5"]["rank"]), "csvB": int(_r_and["d3"]["rank"]),
            "spacer": _r_and["spacer_label"], "spacerLen": len(_r_and["spacer_seq"]),
            "LA": _r_and["l_a"], "LB": _r_and["l_b"],
            "trigLenA": len(_r_and["d5"]["trigger"]), "trigLenB": len(_r_and["d3"]["trigger"]),
            "on": _r_and["on_pct"], "onMin": _r_and["on_min_pct"], "off": _r_and["off_pct"], "offSource": _r_and["off_source"],
            "diff": _r_and["diff_pp"], "diffMin": _r_and["diff_min_pp"], "ratio": _r_and["ratio"], "dG": _r_and["dg"],
            "ctrl": _r_and["ctrl_pct"],
            "openAlone": 100.0 * _r_and["open"]["alone"] / _r_and["l_min"], "openA": 100.0 * _r_and["open"]["+A"] / _r_and["l_min"],
            "openB": 100.0 * _r_and["open"]["+B"] / _r_and["l_min"],
            "ok": [bool(_r_and["ok"][_s]) for _s in _STATES],
            "detail": _r_and["our_rank"] <= ARTIFACT_DETAIL_N,
        })
    (_art_dir / "rows.json").write_text(json.dumps(_rows))
    _shard = 4
    for _i in range(0, min(ARTIFACT_DETAIL_N, len(scored_and)), _shard):
        _chunk = {}
        for _r_and in scored_and[_i:_i + _shard]:
            _chunk[_r_and["our_rank"]] = _detail_panel_html(_r_and, dpi=75).replace(
                "<a class='back' href='#results'>&uarr; back to the table</a>", "")
        (_art_dir / "d" / f"s{_i // _shard}.js").write_text(
            "window.DET=window.DET||{};Object.assign(window.DET," + json.dumps(_chunk) + ");")
    _size = sum(f.stat().st_size for f in _art_dir.rglob("*") if f.is_file())
    print(f"artifact data: {len(_rows)} rows, details for {min(ARTIFACT_DETAIL_N, len(scored_and))} candidates, "
          f"{_size / 1e6:.1f} MB in {_art_dir}")
